# Speech-To-Text Somali Models Comparison

## Whisper Models 

### Regular OpenAI Whisper Model

In [ ]:
# """
# Dual Audio Transcription System:
# 1. AudioTranscriber - Original OpenAI Whisper with GPU support
# 2. SomaliAudioTranscriber - HuggingFace Somali-specialized model
# """

# import os
# import glob
# import shutil
# import subprocess
# import logging
# from datetime import datetime
# from pathlib import Path
# from typing import List, Optional, Union

# import torch

# # Configure logging
# logging.basicConfig(
#     level=logging.INFO,
#     format='%(asctime)s - %(levelname)s - %(message)s',
#     handlers=[
#         logging.FileHandler('audio_transcription.log'),
#         logging.StreamHandler()
#     ]
# )
# logger = logging.getLogger(__name__)


# class AudioTranscriber:
#     """
#     Original AudioTranscriber using OpenAI Whisper with GPU support.
    
#     Handles various audio formats, automatic format conversion, and batch processing
#     with proper error handling, logging, and GPU acceleration.
#     """
    
#     # Supported audio formats
#     SUPPORTED_FORMATS = ["wav", "mp3", "m4a", "flac", "ogg", "webm", "mp4"]
    
#     # Whisper model sizes (from smallest to largest)
#     MODEL_SIZES = ["tiny", "base", "small", "medium", "large", "large-v2", "large-v3"]
    
#     def __init__(self, model_size: str = "small", output_base_dir: Optional[str] = None, 
#                  use_gpu: bool = True):
#         """
#         Initialize the audio transcriber with GPU support.
        
#         Args:
#             model_size: Whisper model size to use
#             output_base_dir: Base directory for transcripts
#             use_gpu: Whether to use GPU acceleration if available
#         """
#         if model_size not in self.MODEL_SIZES:
#             logger.warning(f"Unknown model size '{model_size}'. Using 'small' instead.")
#             model_size = "small"
            
#         self.model_size = model_size
#         self.model = None
#         self.use_gpu = use_gpu
#         self.device = self._setup_device()
        
#         if output_base_dir is None:
#             self.output_base_dir = self._find_project_root() / "data" / "02_intermediate" / "transcripts" / "whisper"
#         else:
#             self.output_base_dir = Path(output_base_dir)
            
#         self._ensure_dependencies()
    
#     def _find_project_root(self) -> Path:
#         """Find the project root directory."""
#         current = Path.cwd()
#         project_name = "somali-radios-with-ai-for-food-security"
        
#         while current != current.parent:
#             if current.name == project_name:
#                 return current
#             if (current / project_name).exists():
#                 return current / project_name
#             current = current.parent
        
#         project_root = Path.cwd() / project_name
#         logger.info(f"Project root not found, using: {project_root}")
#         return project_root
    
#     def _ensure_dependencies(self):
#         """Ensure all required dependencies are installed."""
#         self._check_whisper()
#         self._check_ffmpeg()
        
#     def _check_whisper(self):
#         """Check if Whisper is installed and install if needed."""
#         try:
#             import whisper
#             logger.info("OpenAI Whisper is already available")
#         except ImportError:
#             logger.info("Installing OpenAI Whisper...")
#             subprocess.run(["pip", "install", "openai-whisper"], check=True)
#             logger.info("OpenAI Whisper installed successfully")
            
#     def _check_ffmpeg(self):
#         """Check if ffmpeg is installed and install if needed."""
#         try:
#             result = subprocess.run(['ffmpeg', '-version'], 
#                                   stdout=subprocess.PIPE, 
#                                   stderr=subprocess.PIPE)
#             if result.returncode == 0:
#                 logger.info("ffmpeg is available")
#             else:
#                 raise FileNotFoundError
#         except FileNotFoundError:
#             logger.info("Installing ffmpeg...")
#             try:
#                 subprocess.run(["apt-get", "update", "-qq"], check=True)
#                 subprocess.run(["apt-get", "install", "-qq", "ffmpeg"], check=True)
#                 logger.info("ffmpeg installed successfully via apt-get")
#             except subprocess.CalledProcessError:
#                 logger.error("Failed to install ffmpeg. Please install it manually.")
#                 raise
    
#     def _setup_device(self) -> str:
#         """Setup computing device (GPU/CPU) for Whisper model."""
#         if self.use_gpu and torch.cuda.is_available():
#             device = "cuda"
#             gpu_name = torch.cuda.get_device_name(0)
#             gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
#             logger.info(f"GPU detected: {gpu_name} ({gpu_memory:.1f}GB)")
#             logger.info(f"Using GPU acceleration")
#         else:
#             device = "cpu"
#             if self.use_gpu:
#                 logger.warning("GPU requested but not available. Using CPU instead.")
#             else:
#                 logger.info("Using CPU for transcription")
        
#         return device
    
#     def _load_model(self):
#         """Load the OpenAI Whisper model with GPU support if available."""
#         if self.model is None:
#             logger.info(f"Loading OpenAI Whisper model '{self.model_size}' on {self.device}")
            
#             import whisper
#             self.model = whisper.load_model(self.model_size, device=self.device)
            
#             if self.device == "cuda":
#                 gpu_memory_used = torch.cuda.memory_allocated() / 1024**3
#                 logger.info(f"Model loaded on GPU. GPU memory used: {gpu_memory_used:.2f}GB")
#             else:
#                 logger.info("Model loaded on CPU")
    
#     def _get_audio_files(self, input_dir: Path, audio_formats: Optional[List[str]] = None) -> List[Path]:
#         """Get all audio files in the specified directory."""
#         if audio_formats is None:
#             audio_formats = self.SUPPORTED_FORMATS
            
#         audio_files = []
#         for fmt in audio_formats:
#             pattern = input_dir / f"*.{fmt}"
#             audio_files.extend(glob.glob(str(pattern)))
            
#         return [Path(f) for f in sorted(audio_files)]
    
#     def _convert_audio_to_wav(self, audio_file: Path, temp_dir: Path) -> Optional[Path]:
#         """Convert audio file to WAV format for better compatibility."""
#         temp_dir.mkdir(parents=True, exist_ok=True)
#         temp_wav = temp_dir / f"{audio_file.stem}.wav"
        
#         try:
#             cmd = [
#                 'ffmpeg', '-y', '-i', str(audio_file),
#                 '-ar', '16000',  # 16kHz sample rate
#                 '-ac', '1',      # Mono audio
#                 '-c:a', 'pcm_s16le',  # 16-bit PCM
#                 str(temp_wav)
#             ]
            
#             result = subprocess.run(
#                 cmd,
#                 stdout=subprocess.PIPE,
#                 stderr=subprocess.PIPE,
#                 text=True
#             )
            
#             if result.returncode == 0:
#                 logger.info(f"Successfully converted to WAV: {temp_wav}")
#                 return temp_wav
#             else:
#                 logger.error(f"Conversion failed: {result.stderr}")
#                 return None
                
#         except Exception as e:
#             logger.error(f"Error converting {audio_file}: {e}")
#             return None
    
#     def transcribe_file(self, audio_file: Path, output_dir: Path, language: str = "so") -> Optional[Path]:
#         """Transcribe a single audio file with GPU optimization."""
#         self._load_model()
        
#         base_name = audio_file.stem
#         transcript_file = output_dir / f"{base_name}.txt"
        
#         logger.info(f"Transcribing: {audio_file.name} on {self.device}")
        
#         transcribe_options = {
#             "language": language,
#             "temperature": 0.2,
#             "word_timestamps": True,
#             "verbose": False
#         }
        
#         if self.device == "cuda":
#             transcribe_options["fp16"] = True
#         else:
#             transcribe_options["fp16"] = False
        
#         try:
#             if self.device == "cuda":
#                 torch.cuda.empty_cache()
            
#             result = self.model.transcribe(str(audio_file), **transcribe_options)
            
#             with open(transcript_file, "w", encoding="utf-8") as f:
#                 f.write(result["text"].strip())
            
#             logger.info(f"Transcript saved: {transcript_file}")
#             return transcript_file
            
#         except Exception as e:
#             logger.warning(f"Direct transcription failed for {audio_file.name}: {e}")
            
#             temp_dir = output_dir / "_temp_conversion"
#             temp_wav = self._convert_audio_to_wav(audio_file, temp_dir)
            
#             if temp_wav and temp_wav.exists():
#                 try:
#                     if self.device == "cuda":
#                         torch.cuda.empty_cache()
                    
#                     result = self.model.transcribe(str(temp_wav), **transcribe_options)
                    
#                     with open(transcript_file, "w", encoding="utf-8") as f:
#                         f.write(result["text"].strip())
                    
#                     logger.info(f"Transcript saved after conversion: {transcript_file}")
#                     return transcript_file
                    
#                 except Exception as retry_e:
#                     logger.error(f"Transcription failed even after conversion: {retry_e}")
            
#             return None
    
#     def transcribe_single_file_path(self, audio_file_path: Union[str, Path], language: str = "so") -> Optional[Path]:
#         """
#         Transcribe a single audio file from a file path.
        
#         Args:
#             audio_file_path: Path to audio file (absolute or relative to project root)
#             language: Language code for transcription (default: "so" for Somali)
            
#         Returns:
#             Path to the saved transcript file, or None if transcription failed
#         """
#         audio_path = Path(audio_file_path)
#         project_root = self._find_project_root()
#         project_name = "somali-radios-with-ai-for-food-security"
        
#         # Handle absolute paths
#         if audio_path.is_absolute():
#             if not audio_path.exists():
#                 logger.error(f"Audio file not found: {audio_path}")
#                 return None
#         else:
#             # Handle relative paths - try multiple locations
#             path_str = str(audio_path)
#             resolved_path = None
            
#             # 1. If path contains project name, extract the part after it
#             if project_name in path_str:
#                 parts = path_str.split(f"{project_name}/", 1)
#                 if len(parts) > 1:
#                     relative_part = parts[1]
#                     potential_path = project_root / relative_part
#                     if potential_path.exists():
#                         resolved_path = potential_path
            
#             # 2. Try as-is relative to current directory
#             if not resolved_path and audio_path.exists():
#                 resolved_path = audio_path.resolve()
            
#             # 3. Try relative to project root / data / 01_raw
#             if not resolved_path:
#                 filename = audio_path.name
#                 potential_path = project_root / "data" / "01_raw" / filename
#                 if potential_path.exists():
#                     resolved_path = potential_path
            
#             # 4. Try from project root parent (in case we're in a subdirectory)
#             if not resolved_path:
#                 potential_path = project_root.parent / audio_path
#                 if potential_path.exists():
#                     resolved_path = potential_path
            
#             if resolved_path:
#                 audio_path = resolved_path
#             else:
#                 logger.error(f"Audio file not found: {audio_file_path}")
#                 logger.error(f"Tried: {project_root / path_str.split(project_name + '/', 1)[-1] if project_name in path_str else 'N/A'}")
#                 return None
        
#         # Ensure output directory exists
#         self.output_base_dir.mkdir(parents=True, exist_ok=True)
        
#         # Transcribe the file
#         transcript_file = self.transcribe_file(audio_path, self.output_base_dir, language)
        
#         # Clean up temporary conversion directory if it exists
#         temp_dir = self.output_base_dir / "_temp_conversion"
#         if temp_dir.exists():
#             try:
#                 shutil.rmtree(temp_dir)
#                 logger.info("Removed temporary conversion directory")
#             except Exception as e:
#                 logger.warning(f"Failed to remove temporary directory: {e}")
        
#         if transcript_file:
#             logger.info(f"Transcript saved: {transcript_file}")
#         else:
#             logger.error(f"Failed to transcribe: {audio_path}")
        
#         return transcript_file
    
#     def transcribe_directory(self, input_dir: Union[str, Path], output_dir_name: Optional[str] = None,
#                            language: str = "so", audio_formats: Optional[List[str]] = None) -> List[Path]:
#         """Transcribe all audio files in a directory."""
#         input_path = Path(input_dir)
        
#         if not input_path.exists():
#             logger.error(f"Input directory does not exist: {input_path}")
#             return []
        
#         if output_dir_name is None:
#             output_dir_name = f"transcripts_openai_{self.model_size}_{input_path.name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
#         output_dir = self.output_base_dir / output_dir_name
#         output_dir.mkdir(parents=True, exist_ok=True)
        
#         audio_files = self._get_audio_files(input_path, audio_formats)
        
#         if not audio_files:
#             logger.warning(f"No audio files found in {input_path}")
#             return []
        
#         logger.info(f"Found {len(audio_files)} audio files to transcribe")
#         logger.info(f"Using OpenAI Whisper {self.model_size} model")
#         logger.info(f"Output directory: {output_dir}")
        
#         transcript_files = []
#         failed_files = []
        
#         for i, audio_file in enumerate(audio_files, 1):
#             logger.info(f"Processing {i}/{len(audio_files)}: {audio_file.name}")
            
#             result = self.transcribe_file(audio_file, output_dir, language)
#             if result:
#                 transcript_files.append(result)
#             else:
#                 failed_files.append(audio_file)
        
#         # Clean up temporary conversion directory
#         temp_dir = output_dir / "_temp_conversion"
#         if temp_dir.exists():
#             try:
#                 shutil.rmtree(temp_dir)
#                 logger.info("Removed temporary conversion directory")
#             except Exception as e:
#                 logger.warning(f"Failed to remove temporary directory: {e}")
        
#         # Log summary
#         logger.info(f"\nOpenAI Whisper Transcription Summary:")
#         logger.info(f"Model: {self.model_size}")
#         logger.info(f"Total files processed: {len(audio_files)}")
#         logger.info(f"Successfully transcribed: {len(transcript_files)}")
#         logger.info(f"Failed transcriptions: {len(failed_files)}")
        
#         if failed_files:
#             logger.warning("Failed files:")
#             for file_path in failed_files:
#                 logger.warning(f"  - {file_path}")
        
#         return transcript_files
    
#     def get_system_info(self) -> dict:
#         """Get system information for GPU/CPU usage."""
#         info = {
#             "transcriber_type": "OpenAI Whisper",
#             "device": self.device,
#             "model_size": self.model_size,
#             "torch_version": torch.__version__,
#         }
        
#         if torch.cuda.is_available():
#             info.update({
#                 "cuda_available": True,
#                 "cuda_version": torch.version.cuda,
#                 "gpu_count": torch.cuda.device_count(),
#                 "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else None,
#                 "gpu_memory_total": f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB" if torch.cuda.device_count() > 0 else None
#             })
#         else:
#             info["cuda_available"] = False
        
#         return info


# class SomaliAudioTranscriber:
#     """
#     Specialized AudioTranscriber using HuggingFace Somali-trained Whisper model.
    
#     Uses steja/whisper-small-somali for better Somali language recognition.
#     """
    
#     SUPPORTED_FORMATS = ["wav", "mp3", "m4a", "flac", "ogg", "webm", "mp4"]
    
#     AVAILABLE_MODELS = {
#         "somali": "steja/whisper-small-somali",
#         "multilingual": "openai/whisper-small",
#         "large": "openai/whisper-large-v3"
#     }
    
#     def __init__(self, model_type: str = "somali", output_base_dir: Optional[str] = None, 
#                  use_gpu: bool = True):
#         """Initialize the Somali audio transcriber."""
#         if model_type not in self.AVAILABLE_MODELS:
#             logger.warning(f"Unknown model type '{model_type}'. Using 'somali' instead.")
#             model_type = "somali"
            
#         self.model_type = model_type
#         self.model_id = self.AVAILABLE_MODELS[model_type]
#         self.model = None
#         self.processor = None
#         self.transcriber = None
#         self.use_gpu = use_gpu
#         self.device = self._setup_device()
        
#         if output_base_dir is None:
#             # Save to whisper/ directory (same as OpenAI Whisper model)
#             # Note: If both models are used on the same file, the second will overwrite the first
#             # To avoid this, you can specify a custom output_base_dir
#             self.output_base_dir = self._find_project_root() / "data" / "02_intermediate" / "transcripts" / "whisper"
#         else:
#             self.output_base_dir = Path(output_base_dir)
            
#         self._ensure_dependencies()
    
#     def _find_project_root(self) -> Path:
#         """Find the project root directory."""
#         current = Path.cwd()
#         project_name = "somali-radios-with-ai-for-food-security"
        
#         while current != current.parent:
#             if current.name == project_name:
#                 return current
#             if (current / project_name).exists():
#                 return current / project_name
#             current = current.parent
        
#         project_root = Path.cwd() / project_name
#         logger.info(f"Project root not found, using: {project_root}")
#         return project_root
    
#     def _ensure_dependencies(self):
#         """Ensure all required dependencies are installed."""
#         dependencies = ["transformers", "torch", "torchaudio", "soundfile"]
#         for dep in dependencies:
#             try:
#                 __import__(dep)
#                 logger.info(f"{dep} is already available")
#             except ImportError:
#                 logger.info(f"Installing {dep}...")
#                 subprocess.run(["pip", "install", dep], check=True)
#                 logger.info(f"{dep} installed successfully")
    
#     def _setup_device(self) -> str:
#         """Setup computing device (GPU/CPU) for model."""
#         if self.use_gpu and torch.cuda.is_available():
#             device = "cuda"
#             gpu_name = torch.cuda.get_device_name(0)
#             gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
#             logger.info(f"GPU detected: {gpu_name} ({gpu_memory:.1f}GB)")
#             logger.info(f"Using GPU acceleration")
#         else:
#             device = "cpu"
#             if self.use_gpu:
#                 logger.warning("GPU requested but not available. Using CPU instead.")
#             else:
#                 logger.info("Using CPU for transcription")
        
#         return device
    
#     def _load_model(self):
#         """Load the HuggingFace Whisper model."""
#         if self.transcriber is None:
#             logger.info(f"Loading HuggingFace model '{self.model_id}' on {self.device}")
            
#             from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor
            
#             try:
#                 self.model = AutoModelForSpeechSeq2Seq.from_pretrained(self.model_id)
#                 self.processor = AutoProcessor.from_pretrained(self.model_id)
                
#                 self.model.to(self.device)
                
#                 self.transcriber = pipeline(
#                     "automatic-speech-recognition",
#                     model=self.model,
#                     tokenizer=self.processor.tokenizer,
#                     feature_extractor=self.processor.feature_extractor,
#                     max_new_tokens=128,
#                     chunk_length_s=30,
#                     batch_size=16 if self.device == "cuda" else 8,
#                     device=self.device
#                 )
                
#                 if self.device == "cuda":
#                     gpu_memory_used = torch.cuda.memory_allocated() / 1024**3
#                     logger.info(f"Model loaded on GPU. GPU memory used: {gpu_memory_used:.2f}GB")
#                 else:
#                     logger.info("Model loaded on CPU")
                    
#             except Exception as e:
#                 logger.error(f"Failed to load model {self.model_id}: {e}")
#                 if self.model_type == "somali":
#                     logger.info("Falling back to multilingual model...")
#                     self.model_id = self.AVAILABLE_MODELS["multilingual"]
#                     self.model_type = "multilingual"
#                     self._load_model()
#                 else:
#                     raise
    
#     def _get_audio_files(self, input_dir: Path, audio_formats: Optional[List[str]] = None) -> List[Path]:
#         """Get all audio files in the specified directory."""
#         if audio_formats is None:
#             audio_formats = self.SUPPORTED_FORMATS
            
#         audio_files = []
#         for fmt in audio_formats:
#             pattern = input_dir / f"*.{fmt}"
#             audio_files.extend(glob.glob(str(pattern)))
            
#         return [Path(f) for f in sorted(audio_files)]
    
#     def _convert_audio_to_wav(self, audio_file: Path, temp_dir: Path) -> Optional[Path]:
#         """Convert audio file to WAV format using torchaudio."""
#         temp_dir.mkdir(parents=True, exist_ok=True)
#         temp_wav = temp_dir / f"{audio_file.stem}.wav"
        
#         try:
#             import torchaudio
#             waveform, sample_rate = torchaudio.load(str(audio_file))
#             torchaudio.save(str(temp_wav), waveform, sample_rate)
#             logger.info(f"Successfully converted to WAV: {temp_wav}")
#             return temp_wav
#         except Exception as e:
#             logger.error(f"Error converting {audio_file}: {e}")
#             return None
    
#     def transcribe_file(self, audio_file: Path, output_dir: Path, language: str = "so") -> Optional[Path]:
#         """Transcribe a single audio file using HuggingFace model."""
#         self._load_model()
        
#         base_name = audio_file.stem
#         transcript_file = output_dir / f"{base_name}.txt"
#         file_format = audio_file.suffix[1:].lower()
        
#         logger.info(f"Transcribing: {audio_file.name} (Format: {file_format}) on {self.device}")
        
#         try:
#             if self.device == "cuda":
#                 torch.cuda.empty_cache()
            
#             result = self.transcriber(str(audio_file))
            
#             if isinstance(result, dict) and "text" in result:
#                 transcript_text = result["text"]
#             else:
#                 transcript_text = str(result)
            
#             with open(transcript_file, "w", encoding="utf-8") as f:
#                 f.write(transcript_text.strip())
            
#             logger.info(f"Transcript saved: {transcript_file}")
#             return transcript_file
            
#         except Exception as e:
#             logger.warning(f"Direct transcription failed for {audio_file.name}: {e}")
            
#             if file_format != "wav":
#                 temp_dir = output_dir / "_temp_conversion"
#                 temp_wav = self._convert_audio_to_wav(audio_file, temp_dir)
                
#                 if temp_wav and temp_wav.exists():
#                     try:
#                         if self.device == "cuda":
#                             torch.cuda.empty_cache()
                        
#                         result = self.transcriber(str(temp_wav))
                        
#                         if isinstance(result, dict) and "text" in result:
#                             transcript_text = result["text"]
#                         else:
#                             transcript_text = str(result)
                        
#                         with open(transcript_file, "w", encoding="utf-8") as f:
#                             f.write(transcript_text.strip())
                        
#                         logger.info(f"Transcript saved after conversion: {transcript_file}")
#                         return transcript_file
                        
#                     except Exception as retry_e:
#                         logger.error(f"Transcription failed even after conversion: {retry_e}")
            
#             return None
    
#     def transcribe_single_file_path(self, audio_file_path: Union[str, Path], language: str = "so") -> Optional[Path]:
#         """
#         Transcribe a single audio file from a file path.
        
#         Args:
#             audio_file_path: Path to audio file (absolute or relative to project root)
#             language: Language code for transcription (default: "so" for Somali)
            
#         Returns:
#             Path to the saved transcript file, or None if transcription failed
#         """
#         audio_path = Path(audio_file_path)
#         project_root = self._find_project_root()
#         project_name = "somali-radios-with-ai-for-food-security"
        
#         # Handle absolute paths
#         if audio_path.is_absolute():
#             if not audio_path.exists():
#                 logger.error(f"Audio file not found: {audio_path}")
#                 return None
#         else:
#             # Handle relative paths - try multiple locations
#             path_str = str(audio_path)
#             resolved_path = None
            
#             # 1. If path contains project name, extract the part after it
#             if project_name in path_str:
#                 parts = path_str.split(f"{project_name}/", 1)
#                 if len(parts) > 1:
#                     relative_part = parts[1]
#                     potential_path = project_root / relative_part
#                     if potential_path.exists():
#                         resolved_path = potential_path
            
#             # 2. Try as-is relative to current directory
#             if not resolved_path and audio_path.exists():
#                 resolved_path = audio_path.resolve()
            
#             # 3. Try relative to project root / data / 01_raw
#             if not resolved_path:
#                 filename = audio_path.name
#                 potential_path = project_root / "data" / "01_raw" / filename
#                 if potential_path.exists():
#                     resolved_path = potential_path
            
#             # 4. Try from project root parent (in case we're in a subdirectory)
#             if not resolved_path:
#                 potential_path = project_root.parent / audio_path
#                 if potential_path.exists():
#                     resolved_path = potential_path
            
#             if resolved_path:
#                 audio_path = resolved_path
#             else:
#                 logger.error(f"Audio file not found: {audio_file_path}")
#                 logger.error(f"Tried: {project_root / path_str.split(project_name + '/', 1)[-1] if project_name in path_str else 'N/A'}")
#                 return None
        
#         # Ensure output directory exists
#         self.output_base_dir.mkdir(parents=True, exist_ok=True)
        
#         # Transcribe the file
#         transcript_file = self.transcribe_file(audio_path, self.output_base_dir, language)
        
#         # Clean up temporary conversion directory if it exists
#         temp_dir = self.output_base_dir / "_temp_conversion"
#         if temp_dir.exists():
#             try:
#                 shutil.rmtree(temp_dir)
#                 logger.info("Removed temporary conversion directory")
#             except Exception as e:
#                 logger.warning(f"Failed to remove temporary directory: {e}")
        
#         if transcript_file:
#             logger.info(f"Transcript saved: {transcript_file}")
#         else:
#             logger.error(f"Failed to transcribe: {audio_path}")
        
#         return transcript_file
    
#     def transcribe_directory(self, input_dir: Union[str, Path], output_dir_name: Optional[str] = None,
#                            language: str = "so", audio_formats: Optional[List[str]] = None) -> List[Path]:
#         """Transcribe all audio files in a directory."""
#         input_path = Path(input_dir)
        
#         if not input_path.exists():
#             logger.error(f"Input directory does not exist: {input_path}")
#             return []
        
#         if output_dir_name is None:
#             output_dir_name = f"transcripts_somali_{self.model_type}_{input_path.name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
#         output_dir = self.output_base_dir / output_dir_name
#         output_dir.mkdir(parents=True, exist_ok=True)
        
#         audio_files = self._get_audio_files(input_path, audio_formats)
        
#         if not audio_files:
#             logger.warning(f"No audio files found in {input_path}")
#             return []
        
#         logger.info(f"Found {len(audio_files)} audio files to transcribe")
#         logger.info(f"Using Somali model: {self.model_id}")
#         logger.info(f"Output directory: {output_dir}")
        
#         transcript_files = []
#         failed_files = []
        
#         for i, audio_file in enumerate(audio_files, 1):
#             logger.info(f"Processing {i}/{len(audio_files)}: {audio_file.name}")
            
#             result = self.transcribe_file(audio_file, output_dir, language)
#             if result:
#                 transcript_files.append(result)
#             else:
#                 failed_files.append(audio_file)
        
#         # Clean up temporary conversion directory
#         temp_dir = output_dir / "_temp_conversion"
#         if temp_dir.exists():
#             try:
#                 shutil.rmtree(temp_dir)
#                 logger.info("Removed temporary conversion directory")
#             except Exception as e:
#                 logger.warning(f"Failed to remove temporary directory: {e}")
        
#         # Log summary
#         logger.info(f"\nSomali Model Transcription Summary:")
#         logger.info(f"Model: {self.model_id}")
#         logger.info(f"Total files processed: {len(audio_files)}")
#         logger.info(f"Successfully transcribed: {len(transcript_files)}")
#         logger.info(f"Failed transcriptions: {len(failed_files)}")
        
#         if failed_files:
#             logger.warning("Failed files:")
#             for file_path in failed_files:
#                 logger.warning(f"  - {file_path}")
        
#         return transcript_files
    
#     def get_system_info(self) -> dict:
#         """Get system information."""
#         info = {
#             "transcriber_type": "HuggingFace Somali Whisper",
#             "device": self.device,
#             "model_type": self.model_type,
#             "model_id": self.model_id,
#             "torch_version": torch.__version__,
#         }
        
#         if torch.cuda.is_available():
#             info.update({
#                 "cuda_available": True,
#                 "cuda_version": torch.version.cuda,
#                 "gpu_count": torch.cuda.device_count(),
#                 "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else None,
#                 "gpu_memory_total": f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB" if torch.cuda.device_count() > 0 else None
#             })
#         else:
#             info["cuda_available"] = False
        
#         return info


# # Convenience functions for OpenAI Whisper (original)
# def transcribe_audio_file(audio_file_path: Union[str, Path], language: str = "so",
#                           model_size: str = "small", use_gpu: bool = True) -> Optional[Path]:
#     """
#     Transcribe a single audio file using OpenAI Whisper.
    
#     Args:
#         audio_file_path: Path to audio file (absolute or relative)
#         language: Language code for transcription (default: "so" for Somali)
#         model_size: Whisper model size to use
#         use_gpu: Whether to use GPU acceleration if available
        
#     Returns:
#         Path to the saved transcript file, or None if transcription failed
        
#     Example:
#         transcript_path = transcribe_audio_file(
#             "somali-radios-with-ai-for-food-security/data/01_raw/IDAACADDA 01-JAN-2022.mp3",
#             language="so",
#             model_size="small",
#             use_gpu=True
#         )
#         # Transcript will be saved to:
#         # somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/whisper/IDAACADDA 01-JAN-2022.txt
#     """
#     transcriber = AudioTranscriber(model_size=model_size, use_gpu=use_gpu)
#     system_info = transcriber.get_system_info()
#     logger.info(f"System Info: {system_info}")
#     return transcriber.transcribe_single_file_path(audio_file_path, language)


# def transcribe_audio_files(input_dir: Union[str, Path], output_dir: Optional[str] = None, 
#                           language: str = "so", audio_formats: Optional[List[str]] = None,
#                           model_size: str = "small", use_gpu: bool = True) -> List[Path]:
#     """Original OpenAI Whisper transcription function."""
#     transcriber = AudioTranscriber(model_size=model_size, use_gpu=use_gpu)
#     system_info = transcriber.get_system_info()
#     logger.info(f"System Info: {system_info}")
#     return transcriber.transcribe_directory(input_dir, output_dir, language, audio_formats)


# def transcribe_downloaded_audio(download_dir: Union[str, Path], language: str = "so", 
#                                model_size: str = "small", use_gpu: bool = True) -> List[Path]:
#     """Original OpenAI Whisper download transcription function."""
#     transcriber = AudioTranscriber(model_size=model_size, use_gpu=use_gpu)
#     system_info = transcriber.get_system_info()
#     logger.info(f"System Info: {system_info}")
    
#     project_root = transcriber._find_project_root()
#     download_path = Path(download_dir)
#     if not download_path.is_absolute():
#         download_path = project_root / "data" / "01_raw" / "comparison" / download_dir
    
#     if not download_path.exists():
#         logger.error(f"Download directory not found: {download_path}")
#         return []
    
#     output_name = f"transcripts_{download_path.name}"
#     return transcriber.transcribe_directory(download_path, output_name, language)


# # Convenience functions for Somali specialized model
# def transcribe_audio_file_somali(audio_file_path: Union[str, Path], language: str = "so",
#                                  model_type: str = "somali", use_gpu: bool = True) -> Optional[Path]:
#     """
#     Transcribe a single audio file using Somali specialized Whisper model.
    
#     Args:
#         audio_file_path: Path to audio file (absolute or relative)
#         language: Language code for transcription (default: "so" for Somali)
#         model_type: Model type to use ("somali", "multilingual", "large")
#         use_gpu: Whether to use GPU acceleration if available
        
#     Returns:
#         Path to the saved transcript file, or None if transcription failed
        
#     Example:
#         transcript_path = transcribe_audio_file_somali(
#             "somali-radios-with-ai-for-food-security/data/01_raw/IDAACADDA 01-JAN-2022.mp3",
#             language="so",
#             model_type="somali",
#             use_gpu=True
#         )
#         # Transcript will be saved to:
#         # somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/whisper/IDAACADDA 01-JAN-2022.txt
#     """
#     transcriber = SomaliAudioTranscriber(model_type=model_type, use_gpu=use_gpu)
#     system_info = transcriber.get_system_info()
#     logger.info(f"System Info: {system_info}")
#     return transcriber.transcribe_single_file_path(audio_file_path, language)


# def transcribe_audio_files_somali(input_dir: Union[str, Path], output_dir: Optional[str] = None, 
#                                  language: str = "so", audio_formats: Optional[List[str]] = None,
#                                  model_type: str = "somali", use_gpu: bool = True) -> List[Path]:
#     """Somali specialized Whisper transcription function."""
#     transcriber = SomaliAudioTranscriber(model_type=model_type, use_gpu=use_gpu)
#     system_info = transcriber.get_system_info()
#     logger.info(f"System Info: {system_info}")
#     return transcriber.transcribe_directory(input_dir, output_dir, language, audio_formats)


# def transcribe_downloaded_audio_somali(download_dir: Union[str, Path], language: str = "so", 
#                                       model_type: str = "somali", use_gpu: bool = True) -> List[Path]:
#     """Somali specialized download transcription function."""
#     transcriber = SomaliAudioTranscriber(model_type=model_type, use_gpu=use_gpu)
#     system_info = transcriber.get_system_info()
#     logger.info(f"System Info: {system_info}")
    
#     project_root = transcriber._find_project_root()
#     download_path = Path(download_dir)
#     if not download_path.is_absolute():
#         download_path = project_root / "data" / "01_raw" / "comparison" / download_dir
    
#     if not download_path.exists():
#         logger.error(f"Download directory not found: {download_path}")
#         return []
    
#     output_name = f"transcripts_somali_{download_path.name}"
#     return transcriber.transcribe_directory(download_path, output_name, language)


In [ ]:
"""
OpenAI Whisper Audio Transcription System.

Handles audio transcription using OpenAI's Whisper model with GPU support,
automatic format conversion, and batch processing.
"""

import os
import glob
import shutil
import subprocess
import logging
from datetime import datetime
from pathlib import Path
from typing import List, Optional, Union

import torch

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('whisper_transcription.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


class WhisperTranscriber:
    """
    AudioTranscriber using OpenAI Whisper with GPU support.
    
    Handles various audio formats, automatic format conversion, and batch processing
    with proper error handling, logging, and GPU acceleration.
    """
    
    SUPPORTED_FORMATS = ["wav", "mp3", "m4a", "flac", "ogg", "webm", "mp4"]
    MODEL_SIZES = ["tiny", "base", "small", "medium", "large", "large-v2", "large-v3"]
    
    def __init__(
        self, 
        model_size: str = "small", 
        output_base_dir: Optional[str] = None, 
        use_gpu: bool = True
    ) -> None:
        """
        Initialize the Whisper transcriber with GPU support.
        
        Args:
            model_size: Whisper model size (tiny/base/small/medium/large/large-v2/large-v3)
            output_base_dir: Base directory for transcripts (defaults to project structure)
            use_gpu: Whether to use GPU acceleration if available
            
        Raises:
            subprocess.CalledProcessError: If ffmpeg installation fails
        """
        if model_size not in self.MODEL_SIZES:
            logger.warning(f"Unknown model size '{model_size}'. Using 'small' instead.")
            model_size = "small"
            
        self.model_size = model_size
        self.model = None
        self.use_gpu = use_gpu
        self.device = self._setup_device()
        
        if output_base_dir is None:
            # Separate directory for OpenAI Whisper transcripts
            self.output_base_dir = (
                self._find_project_root() / "data" / "02_intermediate" / 
                "transcripts" / "openai_whisper"
            )
        else:
            self.output_base_dir = Path(output_base_dir)
            
        self._ensure_dependencies()
    
    def _find_project_root(self) -> Path:
        """
        Find the project root directory by searching for project name.
        
        Returns:
            Path to project root directory
        """
        current = Path.cwd()
        project_name = "somali-radios-with-ai-for-food-security"
        
        while current != current.parent:
            if current.name == project_name:
                return current
            if (current / project_name).exists():
                return current / project_name
            current = current.parent
        
        project_root = Path.cwd() / project_name
        logger.info(f"Project root not found, using: {project_root}")
        return project_root
    
    def _ensure_dependencies(self) -> None:
        """Ensure all required dependencies are installed."""
        self._check_whisper()
        self._check_ffmpeg()
        
    def _check_whisper(self) -> None:
        """Check if Whisper is installed and install if needed."""
        try:
            import whisper
            logger.info("OpenAI Whisper is already available")
        except ImportError:
            logger.info("Installing OpenAI Whisper...")
            subprocess.run(["pip", "install", "openai-whisper"], check=True)
            logger.info("OpenAI Whisper installed successfully")
            
    def _check_ffmpeg(self) -> None:
        """Check if ffmpeg is installed and install if needed."""
        try:
            result = subprocess.run(
                ['ffmpeg', '-version'], 
                stdout=subprocess.PIPE, 
                stderr=subprocess.PIPE
            )
            if result.returncode == 0:
                logger.info("ffmpeg is available")
            else:
                raise FileNotFoundError
        except FileNotFoundError:
            logger.info("Installing ffmpeg...")
            try:
                subprocess.run(["apt-get", "update", "-qq"], check=True)
                subprocess.run(["apt-get", "install", "-qq", "ffmpeg"], check=True)
                logger.info("ffmpeg installed successfully via apt-get")
            except subprocess.CalledProcessError:
                logger.error("Failed to install ffmpeg. Please install it manually.")
                raise
    
    def _setup_device(self) -> str:
        """
        Setup computing device (GPU/CPU) for Whisper model.
        
        Returns:
            Device string ("cuda" or "cpu")
        """
        if self.use_gpu and torch.cuda.is_available():
            device = "cuda"
            gpu_name = torch.cuda.get_device_name(0)
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
            logger.info(f"GPU detected: {gpu_name} ({gpu_memory:.1f}GB)")
            logger.info(f"Using GPU acceleration")
        else:
            device = "cpu"
            if self.use_gpu:
                logger.warning("GPU requested but not available. Using CPU instead.")
            else:
                logger.info("Using CPU for transcription")
        
        return device
    
    def _load_model(self) -> None:
        """Load the OpenAI Whisper model with GPU support if available."""
        if self.model is None:
            logger.info(f"Loading OpenAI Whisper model '{self.model_size}' on {self.device}")
            
            import whisper
            self.model = whisper.load_model(self.model_size, device=self.device)
            
            if self.device == "cuda":
                gpu_memory_used = torch.cuda.memory_allocated() / 1024**3
                logger.info(f"Model loaded on GPU. GPU memory used: {gpu_memory_used:.2f}GB")
            else:
                logger.info("Model loaded on CPU")
    
    def _get_audio_files(
        self, 
        input_dir: Path, 
        audio_formats: Optional[List[str]] = None
    ) -> List[Path]:
        """
        Get all audio files in the specified directory.
        
        Args:
            input_dir: Directory containing audio files
            audio_formats: List of file extensions to search for
            
        Returns:
            Sorted list of audio file paths
        """
        if audio_formats is None:
            audio_formats = self.SUPPORTED_FORMATS
            
        audio_files = []
        for fmt in audio_formats:
            pattern = input_dir / f"*.{fmt}"
            audio_files.extend(glob.glob(str(pattern)))
            
        return [Path(f) for f in sorted(audio_files)]
    
    def _convert_audio_to_wav(self, audio_file: Path, temp_dir: Path) -> Optional[Path]:
        """
        Convert audio file to WAV format for better compatibility.
        
        Args:
            audio_file: Path to input audio file
            temp_dir: Directory for temporary converted file
            
        Returns:
            Path to converted WAV file, or None if conversion failed
        """
        temp_dir.mkdir(parents=True, exist_ok=True)
        temp_wav = temp_dir / f"{audio_file.stem}.wav"
        
        try:
            cmd = [
                'ffmpeg', '-y', '-i', str(audio_file),
                '-ar', '16000',  # 16kHz sample rate (Whisper standard)
                '-ac', '1',      # Mono audio
                '-c:a', 'pcm_s16le',  # 16-bit PCM
                str(temp_wav)
            ]
            
            result = subprocess.run(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            
            if result.returncode == 0:
                logger.info(f"Successfully converted to WAV: {temp_wav}")
                return temp_wav
            else:
                logger.error(f"Conversion failed: {result.stderr}")
                return None
                
        except Exception as e:
            logger.error(f"Error converting {audio_file}: {e}")
            return None
    
    def transcribe_file(
        self, 
        audio_file: Path, 
        output_dir: Path, 
        language: str = "so"
    ) -> Optional[Path]:
        """
        Transcribe a single audio file with GPU optimization.
        
        Args:
            audio_file: Path to audio file
            output_dir: Directory to save transcript
            language: Language code (default: "so" for Somali)
            
        Returns:
            Path to saved transcript file, or None if transcription failed
        """
        self._load_model()
        
        base_name = audio_file.stem
        transcript_file = output_dir / f"{base_name}.txt"
        
        logger.info(f"Transcribing: {audio_file.name} on {self.device}")
        
        # Configure transcription options based on device
        transcribe_options = {
            "language": language,
            "temperature": 0.2,
            "word_timestamps": True,
            "verbose": False,
            "fp16": self.device == "cuda"  # Use FP16 only on GPU
        }
        
        try:
            if self.device == "cuda":
                torch.cuda.empty_cache()  # Clear GPU memory before transcription
            
            result = self.model.transcribe(str(audio_file), **transcribe_options)
            
            with open(transcript_file, "w", encoding="utf-8") as f:
                f.write(result["text"].strip())
            
            logger.info(f"Transcript saved: {transcript_file}")
            return transcript_file
            
        except Exception as e:
            logger.warning(f"Direct transcription failed for {audio_file.name}: {e}")
            
            # Try converting to WAV and retrying
            temp_dir = output_dir / "_temp_conversion"
            temp_wav = self._convert_audio_to_wav(audio_file, temp_dir)
            
            if temp_wav and temp_wav.exists():
                try:
                    if self.device == "cuda":
                        torch.cuda.empty_cache()
                    
                    result = self.model.transcribe(str(temp_wav), **transcribe_options)
                    
                    with open(transcript_file, "w", encoding="utf-8") as f:
                        f.write(result["text"].strip())
                    
                    logger.info(f"Transcript saved after conversion: {transcript_file}")
                    return transcript_file
                    
                except Exception as retry_e:
                    logger.error(f"Transcription failed even after conversion: {retry_e}")
            
            return None
    
    def transcribe_single_file_path(
        self, 
        audio_file_path: Union[str, Path], 
        language: str = "so"
    ) -> Optional[Path]:
        """
        Transcribe a single audio file from a file path.
        
        Args:
            audio_file_path: Path to audio file (absolute or relative to project root)
            language: Language code for transcription (default: "so" for Somali)
            
        Returns:
            Path to the saved transcript file, or None if transcription failed
            
        Raises:
            FileNotFoundError: If audio file cannot be located
        """
        audio_path = Path(audio_file_path)
        project_root = self._find_project_root()
        project_name = "somali-radios-with-ai-for-food-security"
        
        # Handle absolute paths
        if audio_path.is_absolute():
            if not audio_path.exists():
                logger.error(f"Audio file not found: {audio_path}")
                return None
        else:
            # Handle relative paths - try multiple locations
            path_str = str(audio_path)
            resolved_path = None
            
            # Extract path after project name if present
            if project_name in path_str:
                parts = path_str.split(f"{project_name}/", 1)
                if len(parts) > 1:
                    relative_part = parts[1]
                    potential_path = project_root / relative_part
                    if potential_path.exists():
                        resolved_path = potential_path
            
            # Try as-is relative to current directory
            if not resolved_path and audio_path.exists():
                resolved_path = audio_path.resolve()
            
            # Try relative to project root / data / 01_raw
            if not resolved_path:
                filename = audio_path.name
                potential_path = project_root / "data" / "01_raw" / filename
                if potential_path.exists():
                    resolved_path = potential_path
            
            # Try from project root parent
            if not resolved_path:
                potential_path = project_root.parent / audio_path
                if potential_path.exists():
                    resolved_path = potential_path
            
            if resolved_path:
                audio_path = resolved_path
            else:
                logger.error(f"Audio file not found: {audio_file_path}")
                return None
        
        # Ensure output directory exists
        self.output_base_dir.mkdir(parents=True, exist_ok=True)
        
        # Transcribe the file
        transcript_file = self.transcribe_file(audio_path, self.output_base_dir, language)
        
        # Clean up temporary conversion directory if it exists
        temp_dir = self.output_base_dir / "_temp_conversion"
        if temp_dir.exists():
            try:
                shutil.rmtree(temp_dir)
                logger.info("Removed temporary conversion directory")
            except Exception as e:
                logger.warning(f"Failed to remove temporary directory: {e}")
        
        if transcript_file:
            logger.info(f"Transcript saved: {transcript_file}")
        else:
            logger.error(f"Failed to transcribe: {audio_path}")
        
        return transcript_file
    
    def transcribe_directory(
        self, 
        input_dir: Union[str, Path], 
        output_dir_name: Optional[str] = None,
        language: str = "so", 
        audio_formats: Optional[List[str]] = None
    ) -> List[Path]:
        """
        Transcribe all audio files in a directory.
        
        Args:
            input_dir: Directory containing audio files
            output_dir_name: Name for output subdirectory (auto-generated if None)
            language: Language code (default: "so" for Somali)
            audio_formats: List of audio formats to process
            
        Returns:
            List of paths to successfully created transcript files
        """
        input_path = Path(input_dir)
        
        if not input_path.exists():
            logger.error(f"Input directory does not exist: {input_path}")
            return []
        
        if output_dir_name is None:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            output_dir_name = f"batch_{self.model_size}_{input_path.name}_{timestamp}"
        
        output_dir = self.output_base_dir / output_dir_name
        output_dir.mkdir(parents=True, exist_ok=True)
        
        audio_files = self._get_audio_files(input_path, audio_formats)
        
        if not audio_files:
            logger.warning(f"No audio files found in {input_path}")
            return []
        
        logger.info(f"Found {len(audio_files)} audio files to transcribe")
        logger.info(f"Using OpenAI Whisper {self.model_size} model")
        logger.info(f"Output directory: {output_dir}")
        
        transcript_files = []
        failed_files = []
        
        for i, audio_file in enumerate(audio_files, 1):
            logger.info(f"Processing {i}/{len(audio_files)}: {audio_file.name}")
            
            result = self.transcribe_file(audio_file, output_dir, language)
            if result:
                transcript_files.append(result)
            else:
                failed_files.append(audio_file)
        
        # Clean up temporary conversion directory
        temp_dir = output_dir / "_temp_conversion"
        if temp_dir.exists():
            try:
                shutil.rmtree(temp_dir)
                logger.info("Removed temporary conversion directory")
            except Exception as e:
                logger.warning(f"Failed to remove temporary directory: {e}")
        
        # Log summary
        logger.info(f"\n=== OpenAI Whisper Transcription Summary ===")
        logger.info(f"Model: {self.model_size}")
        logger.info(f"Total files processed: {len(audio_files)}")
        logger.info(f"Successfully transcribed: {len(transcript_files)}")
        logger.info(f"Failed transcriptions: {len(failed_files)}")
        
        if failed_files:
            logger.warning("Failed files:")
            for file_path in failed_files:
                logger.warning(f"  - {file_path}")
        
        return transcript_files
    
    def get_system_info(self) -> dict:
        """
        Get system information for GPU/CPU usage.
        
        Returns:
            Dictionary containing system and model information
        """
        info = {
            "transcriber_type": "OpenAI Whisper",
            "device": self.device,
            "model_size": self.model_size,
            "torch_version": torch.__version__,
        }
        
        if torch.cuda.is_available():
            info.update({
                "cuda_available": True,
                "cuda_version": torch.version.cuda,
                "gpu_count": torch.cuda.device_count(),
                "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else None,
                "gpu_memory_total": f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB" if torch.cuda.device_count() > 0 else None
            })
        else:
            info["cuda_available"] = False
        
        return info


# Convenience functions
def transcribe_audio_file(
    audio_file_path: Union[str, Path], 
    language: str = "so",
    model_size: str = "small", 
    use_gpu: bool = True
) -> Optional[Path]:
    """
    Transcribe a single audio file using OpenAI Whisper.
    
    Args:
        audio_file_path: Path to audio file (absolute or relative)
        language: Language code for transcription (default: "so" for Somali)
        model_size: Whisper model size to use
        use_gpu: Whether to use GPU acceleration if available
        
    Returns:
        Path to the saved transcript file, or None if transcription failed
    """
    transcriber = WhisperTranscriber(model_size=model_size, use_gpu=use_gpu)
    system_info = transcriber.get_system_info()
    logger.info(f"System Info: {system_info}")
    return transcriber.transcribe_single_file_path(audio_file_path, language)


def transcribe_audio_directory(
    input_dir: Union[str, Path], 
    output_dir: Optional[str] = None, 
    language: str = "so", 
    audio_formats: Optional[List[str]] = None,
    model_size: str = "small", 
    use_gpu: bool = True
) -> List[Path]:
    """
    Transcribe all audio files in a directory using OpenAI Whisper.
    
    Args:
        input_dir: Directory containing audio files
        output_dir: Name for output subdirectory
        language: Language code (default: "so")
        audio_formats: List of audio formats to process
        model_size: Whisper model size
        use_gpu: Whether to use GPU
        
    Returns:
        List of paths to created transcript files
    """
    transcriber = WhisperTranscriber(model_size=model_size, use_gpu=use_gpu)
    system_info = transcriber.get_system_info()
    logger.info(f"System Info: {system_info}")
    return transcriber.transcribe_directory(input_dir, output_dir, language, audio_formats)

In [ ]:
# Example usage for single file transcription
if __name__ == "__main__":
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    audio_file_path = "somali-radios-with-ai-for-food-security/data/01_raw/IDAACADDA 01-JAN-2022.mp3"
    
    # Original OpenAI Whisper approach
    print("\n=== Using Original OpenAI Whisper ===")
    transcript_path = transcribe_audio_file(
        audio_file_path=audio_file_path,
        language="so",
        model_size="large-v3",
        use_gpu=True
    )
    
    if transcript_path:
        print(f"OpenAI Whisper completed! Transcript saved to: {transcript_path}")
    else:
        print("Transcription failed!")

## Evaluation Feature

In [ ]:
"""
Somali Transcript Analysis Tool

A comprehensive tool for analyzing Somali language transcripts with linguistic metrics,
pattern detection, and quality assessment capabilities.
"""

import os
import re
import textwrap
import logging
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Union
from dataclasses import dataclass

# Optional NLTK import with graceful fallback
try:
    import nltk
    from nltk.tokenize import sent_tokenize, word_tokenize
    NLTK_AVAILABLE = True
    
    # Download required NLTK data if needed
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt', quiet=True)
        
except ImportError:
    NLTK_AVAILABLE = False
    logging.info("NLTK not available. Some advanced features will be disabled.")


# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


@dataclass
class TranscriptStats:
    """Data class to hold transcript analysis statistics."""
    num_lines: int
    total_words: int
    total_characters: int
    avg_words_per_line: float
    avg_word_length: float
    avg_chars_per_line: float
    repetition_count: int
    estimated_sentences: int
    line_length_range: Tuple[int, int]
    most_common_words: List[Tuple[str, int]]
    repeated_segments: List[Tuple[str, int]]


class SomaliTranscriptAnalyzer:
    """
    A comprehensive analyzer for Somali language transcripts.
    
    Provides linguistic analysis, quality metrics, and pattern detection
    specifically designed for Somali radio transcripts.
    """
    
    # Common Somali stop words and particles
    SOMALI_STOP_WORDS = {
        'ah', 'aha', 'waa', 'waxaa', 'baa', 'ayaa', 'oo', 'iyo', 'ama', 'laakiin',
        'si', 'ka', 'ku', 'la', 'uu', 'ay', 'aan', 'aad', 'uu', 'ee', 'oo',
        'in', 'an', 'soo', 'wax', 'ma', 'mise', 'haddii', 'markii', 'intii'
    }
    
    # Somali sentence-ending patterns
    SENTENCE_ENDINGS = ['.', '!', '?', ':', ';']
    
    def __init__(self, project_root: Optional[Path] = None):
        """
        Initialize the Somali transcript analyzer.
        
        Args:
            project_root: Path to project root (auto-detected if None)
        """
        if project_root is None:
            self.project_root = self._find_project_root()
        else:
            self.project_root = Path(project_root)
            
        self.transcripts_dir = self.project_root / "data" / "02_intermediate" / "transcripts"
        self.analysis_output_dir = self.project_root / "data" / "03_primary" / "analysis"
        self.analysis_output_dir.mkdir(parents=True, exist_ok=True)
        
    def _find_project_root(self) -> Path:
        """Find the project root directory."""
        current = Path.cwd()
        project_name = "somali-radios-with-ai-for-food-security"
        
        while current != current.parent:
            if current.name == project_name:
                return current
            if (current / project_name).exists():
                return current / project_name
            current = current.parent
        
        # If not found, use current working directory
        logger.warning(f"Project root not found, using current directory: {Path.cwd()}")
        return Path.cwd()
    
    def _read_transcript_file(self, file_path: Path, max_lines: Optional[int] = None) -> List[str]:
        """
        Read transcript file and return non-empty lines.
        
        Args:
            file_path: Path to transcript file
            max_lines: Maximum number of lines to read
            
        Returns:
            List of non-empty lines
        """
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                if max_lines:
                    lines = [file.readline().strip() for _ in range(max_lines)]
                else:
                    lines = [line.strip() for line in file.readlines()]
                    
            # Remove empty lines
            lines = [line for line in lines if line.strip()]
            return lines
            
        except FileNotFoundError:
            logger.error(f"File not found: {file_path}")
            raise
        except Exception as e:
            logger.error(f"Error reading file {file_path}: {e}")
            raise
    
    def _calculate_basic_stats(self, lines: List[str]) -> Dict[str, Any]:
        """Calculate basic statistical metrics for the transcript."""
        if not lines:
            return {}
            
        num_lines = len(lines)
        total_chars = sum(len(line) for line in lines)
        total_words = sum(len(line.split()) for line in lines)
        
        return {
            'num_lines': num_lines,
            'total_words': total_words,
            'total_characters': total_chars,
            'avg_words_per_line': total_words / num_lines,
            'avg_word_length': sum(len(word) for line in lines for word in line.split()) / total_words if total_words > 0 else 0,
            'avg_chars_per_line': total_chars / num_lines
        }
    
    def _analyze_word_frequency(self, lines: List[str], top_n: int = 20) -> List[Tuple[str, int]]:
        """Analyze word frequency in the transcript."""
        # Combine all lines and normalize
        all_text = ' '.join(lines).lower()
        # Remove punctuation and split into words
        words = re.findall(r'\b\w+\b', all_text)
        
        # Filter out very short words and common stop words
        filtered_words = [
            word for word in words 
            if len(word) > 2 and word not in self.SOMALI_STOP_WORDS
        ]
        
        word_freq = Counter(filtered_words)
        return word_freq.most_common(top_n)
    
    def _detect_repetitions(self, lines: List[str]) -> int:
        """Count adjacent word repetitions in the transcript."""
        repetition_count = 0
        for line in lines:
            words = line.split()
            for i in range(len(words) - 1):
                if words[i].lower() == words[i+1].lower():
                    repetition_count += 1
        return repetition_count
    
    def _find_repeated_segments(self, lines: List[str], min_occurrences: int = 3) -> List[Tuple[str, int]]:
        """Find commonly repeated word segments in the transcript."""
        text = ' '.join(lines).lower()
        words = text.split()
        repeated_segments = []
        
        # Check for segments of different lengths
        for segment_len in range(2, 6):  # 2 to 5 words
            if len(words) < segment_len:
                continue
                
            segments = [
                ' '.join(words[i:i+segment_len]) 
                for i in range(len(words) - segment_len + 1)
            ]
            
            segment_counts = Counter(segments)
            repeated = [
                (segment, count) for segment, count in segment_counts.items() 
                if count >= min_occurrences and len(segment.strip()) > 5
            ]
            repeated_segments.extend(repeated)
        
        # Sort by frequency and return top results
        return sorted(repeated_segments, key=lambda x: x[1], reverse=True)[:15]
    
    def _estimate_sentences(self, lines: List[str]) -> Tuple[int, List[str]]:
        """Estimate sentence count and extract sample sentences."""
        sentences = []
        current_sentence = ""
        
        for line in lines:
            # Split by sentence-ending punctuation
            parts = re.split(r'[.!?:;]', line)
            
            for i, part in enumerate(parts):
                if part.strip():
                    if current_sentence:
                        current_sentence += " " + part.strip()
                    else:
                        current_sentence = part.strip()
                    
                    # If this part was followed by punctuation, end the sentence
                    if i < len(parts) - 1 or any(line.rstrip().endswith(p) for p in self.SENTENCE_ENDINGS):
                        if current_sentence:
                            sentences.append(current_sentence)
                            current_sentence = ""
        
        # Add any remaining sentence
        if current_sentence:
            sentences.append(current_sentence)
        
        return len(sentences), sentences[:10]  # Return count and first 10 sentences
    
    def _analyze_line_lengths(self, lines: List[str]) -> Tuple[int, int, List[int]]:
        """Analyze line length distribution."""
        line_lengths = [len(line.split()) for line in lines]
        return min(line_lengths), max(line_lengths), line_lengths
    
    def _assess_transcript_quality(self, stats: TranscriptStats) -> Dict[str, Any]:
        """Assess the quality of the transcript based on various metrics."""
        quality_issues = []
        quality_score = 100  # Start with perfect score
        
        # Check for excessive repetitions
        repetition_ratio = stats.repetition_count / stats.total_words if stats.total_words > 0 else 0
        if repetition_ratio > 0.1:  # More than 10% repetitions
            quality_issues.append(f"High repetition rate: {repetition_ratio:.2%}")
            quality_score -= 20
        
        # Check average word length (very short might indicate poor transcription)
        if stats.avg_word_length < 3:
            quality_issues.append(f"Short average word length: {stats.avg_word_length:.2f}")
            quality_score -= 15
        
        # Check line length variance (too uniform might indicate issues)
        min_len, max_len = stats.line_length_range
        if max_len - min_len < 5 and stats.num_lines > 10:
            quality_issues.append("Low variance in line lengths")
            quality_score -= 10
        
        # Check for very short lines that might indicate transcription errors
        if min_len < 2:
            quality_issues.append("Very short lines detected")
            quality_score -= 10
        
        return {
            'quality_score': max(0, quality_score),  # Don't go below 0
            'quality_issues': quality_issues,
            'assessment': 'Good' if quality_score >= 80 else 'Fair' if quality_score >= 60 else 'Poor'
        }
    
    def analyze_transcript(self, file_path: Union[str, Path], max_lines: Optional[int] = None, 
                          sample_size: int = 10, save_analysis: bool = True) -> TranscriptStats:
        """
        Perform comprehensive analysis of a Somali transcript.
        
        Args:
            file_path: Path to transcript file (absolute or relative to transcripts directory)
            max_lines: Maximum number of lines to analyze
            sample_size: Number of sample lines to display
            save_analysis: Whether to save analysis results to file
            
        Returns:
            TranscriptStats object with analysis results
        """
        # Handle path resolution
        file_path = Path(file_path)
        path_str = str(file_path)
        resolved_path = None
        
        if file_path.is_absolute():
            # Absolute path - just verify it exists
            if file_path.exists():
                resolved_path = file_path
        else:
            # Handle relative paths
            # 1. Strip "transcripts/" prefix if present (user might provide "transcripts/whisper/file.txt")
            if path_str.startswith("transcripts/"):
                path_str = path_str.replace("transcripts/", "", 1)
                file_path = Path(path_str)
            
            # 2. Try as subdirectory path within transcripts_dir (e.g., "whisper/IDAACADDA 01-JAN-2022.txt")
            potential_path = self.transcripts_dir / file_path
            if potential_path.exists():
                resolved_path = potential_path
            # 3. Try as just filename in transcripts_dir
            elif (self.transcripts_dir / file_path.name).exists():
                resolved_path = self.transcripts_dir / file_path.name
            # 4. Try relative to current directory
            elif file_path.exists():
                resolved_path = file_path.resolve()
            # 5. Last resort: search recursively (but warn user)
            else:
                matching_files = list(self.transcripts_dir.rglob(file_path.name))
                if matching_files:
                    if len(matching_files) > 1:
                        logger.warning(f"Multiple files found with name '{file_path.name}'. Using first match: {matching_files[0]}")
                        logger.warning(f"Other matches: {[str(f) for f in matching_files[1:]]}")
                    resolved_path = matching_files[0]
                    logger.info(f"Found file using recursive search: {resolved_path}")
        
        if not resolved_path or not resolved_path.exists():
            raise FileNotFoundError(f"Transcript file not found: {file_path}")
        
        file_path = resolved_path
        logger.info(f"Analyzing transcript: {file_path}")
        logger.info(f"Full path: {file_path.absolute()}")
        
        # Read transcript
        lines = self._read_transcript_file(file_path, max_lines)
        
        if not lines:
            logger.warning("No content found in transcript file")
            return None
        
        # Perform analysis
        basic_stats = self._calculate_basic_stats(lines)
        most_common = self._analyze_word_frequency(lines)
        repetitions = self._detect_repetitions(lines)
        repeated_segments = self._find_repeated_segments(lines)
        sentence_count, sample_sentences = self._estimate_sentences(lines)
        min_len, max_len, line_lengths = self._analyze_line_lengths(lines)
        
        # Create stats object
        stats = TranscriptStats(
            num_lines=basic_stats['num_lines'],
            total_words=basic_stats['total_words'],
            total_characters=basic_stats['total_characters'],
            avg_words_per_line=basic_stats['avg_words_per_line'],
            avg_word_length=basic_stats['avg_word_length'],
            avg_chars_per_line=basic_stats['avg_chars_per_line'],
            repetition_count=repetitions,
            estimated_sentences=sentence_count,
            line_length_range=(min_len, max_len),
            most_common_words=most_common,
            repeated_segments=repeated_segments
        )
        
        # Quality assessment
        quality_info = self._assess_transcript_quality(stats)
        
        # Display results
        self._display_analysis_results(file_path.name, lines, stats, quality_info, 
                                     sample_sentences, sample_size)
        
        # Save analysis if requested
        if save_analysis:
            self._save_analysis_report(file_path, stats, quality_info, sample_sentences)
        
        return stats
    
    def _display_analysis_results(self, filename: str, lines: List[str], stats: TranscriptStats,
                                 quality_info: Dict[str, Any], sample_sentences: List[str],
                                 sample_size: int):
        """Display comprehensive analysis results."""
        print(f"\n{'='*60}")
        print(f"SOMALI TRANSCRIPT ANALYSIS: {filename}")
        print(f"{'='*60}")
        
        # Sample lines
        print(f"\n{'='*20} SAMPLE LINES {'='*20}")
        sample_indices = list(range(min(sample_size, len(lines))))
        for i in sample_indices:
            wrapped_text = textwrap.fill(lines[i], width=80)
            print(f"Line {i+1}: {wrapped_text}")
        
        # Basic statistics
        print(f"\n{'='*20} BASIC STATISTICS {'='*20}")
        print(f"Total lines: {stats.num_lines:,}")
        print(f"Total words: {stats.total_words:,}")
        print(f"Total characters: {stats.total_characters:,}")
        print(f"Average words per line: {stats.avg_words_per_line:.2f}")
        print(f"Average word length: {stats.avg_word_length:.2f} characters")
        print(f"Average characters per line: {stats.avg_chars_per_line:.1f}")
        print(f"Estimated sentences: {stats.estimated_sentences:,}")
        print(f"Adjacent word repetitions: {stats.repetition_count}")
        
        # Line length analysis
        min_len, max_len = stats.line_length_range
        print(f"\nLine length range: {min_len} - {max_len} words")
        
        # Quality assessment
        print(f"\n{'='*20} QUALITY ASSESSMENT {'='*20}")
        print(f"Quality score: {quality_info['quality_score']}/100 ({quality_info['assessment']})")
        if quality_info['quality_issues']:
            print("Quality issues detected:")
            for issue in quality_info['quality_issues']:
                print(f"  - {issue}")
        else:
            print("No significant quality issues detected.")
        
        # Most common words
        print(f"\n{'='*20} MOST COMMON WORDS {'='*20}")
        for word, count in stats.most_common_words[:15]:
            percentage = (count / stats.total_words) * 100
            print(f"{word:15} {count:6,} ({percentage:.1f}%)")
        
        # Repeated segments
        if stats.repeated_segments:
            print(f"\n{'='*20} REPEATED SEGMENTS {'='*20}")
            for segment, count in stats.repeated_segments[:10]:
                print(f"'{segment}' → {count} times")
        
        # Sample sentences
        if sample_sentences:
            print(f"\n{'='*20} SAMPLE SENTENCES {'='*20}")
            for i, sentence in enumerate(sample_sentences[:5], 1):
                wrapped = textwrap.fill(sentence, width=75)
                print(f"{i}. {wrapped}")
    
    def _save_analysis_report(self, file_path: Path, stats: TranscriptStats, 
                             quality_info: Dict[str, Any], sample_sentences: List[str]):
        """Save analysis report to file."""
        report_name = f"analysis_{file_path.stem}_{file_path.parent.name}.txt"
        report_path = self.analysis_output_dir / report_name
        
        try:
            with open(report_path, 'w', encoding='utf-8') as f:
                f.write(f"Somali Transcript Analysis Report\n")
                f.write(f"Generated: {file_path}\n")
                f.write(f"{'='*50}\n\n")
                
                # Write all the analysis data
                f.write(f"BASIC STATISTICS\n")
                f.write(f"Lines: {stats.num_lines:,}\n")
                f.write(f"Words: {stats.total_words:,}\n")
                f.write(f"Characters: {stats.total_characters:,}\n")
                f.write(f"Avg words/line: {stats.avg_words_per_line:.2f}\n")
                f.write(f"Avg word length: {stats.avg_word_length:.2f}\n")
                f.write(f"Estimated sentences: {stats.estimated_sentences:,}\n\n")
                
                f.write(f"QUALITY ASSESSMENT\n")
                f.write(f"Score: {quality_info['quality_score']}/100 ({quality_info['assessment']})\n")
                if quality_info['quality_issues']:
                    f.write("Issues:\n")
                    for issue in quality_info['quality_issues']:
                        f.write(f"  - {issue}\n")
                f.write("\n")
                
                # Save common words and repeated segments
                f.write("MOST COMMON WORDS\n")
                for word, count in stats.most_common_words:
                    f.write(f"{word}: {count}\n")
                
                if stats.repeated_segments:
                    f.write("\nREPEATED SEGMENTS\n")
                    for segment, count in stats.repeated_segments:
                        f.write(f"'{segment}': {count}\n")
            
            logger.info(f"Analysis report saved: {report_path}")
            
        except Exception as e:
            logger.error(f"Failed to save analysis report: {e}")
    
    def analyze_directory(self, transcript_dir: Optional[Union[str, Path]] = None,
                         file_pattern: str = "*.txt") -> Dict[str, TranscriptStats]:
        """
        Analyze all transcript files in a directory.
        
        Args:
            transcript_dir: Directory containing transcript files (default: auto-detect)
            file_pattern: File pattern to match (default: *.txt)
            
        Returns:
            Dictionary mapping filenames to analysis results
        """
        if transcript_dir is None:
            # Find the most recent transcript directory
            transcript_dirs = [d for d in self.transcripts_dir.iterdir() if d.is_dir()]
            if not transcript_dirs:
                logger.error("No transcript directories found")
                return {}
            transcript_dir = max(transcript_dirs, key=lambda d: d.stat().st_mtime)
        else:
            transcript_dir = Path(transcript_dir)
            if not transcript_dir.is_absolute():
                transcript_dir = self.transcripts_dir / transcript_dir
        
        logger.info(f"Analyzing all transcripts in: {transcript_dir}")
        
        # Find all transcript files
        transcript_files = list(transcript_dir.glob(file_pattern))
        
        if not transcript_files:
            logger.warning(f"No transcript files found matching '{file_pattern}' in {transcript_dir}")
            return {}
        
        results = {}
        for transcript_file in transcript_files:
            try:
                stats = self.analyze_transcript(transcript_file, save_analysis=True)
                results[transcript_file.name] = stats
            except Exception as e:
                logger.error(f"Failed to analyze {transcript_file.name}: {e}")
        
        logger.info(f"Completed analysis of {len(results)} transcript files")
        return results


# Convenience functions for easy usage
def analyze_somali_transcript(file_path: Union[str, Path], max_lines: Optional[int] = None,
                             sample_size: int = 10) -> TranscriptStats:
    """
    Convenience function to analyze a single Somali transcript.
    
    Args:
        file_path: Path to transcript file
        max_lines: Maximum number of lines to analyze
        sample_size: Number of sample lines to display
        
    Returns:
        TranscriptStats object with analysis results
    """
    analyzer = SomaliTranscriptAnalyzer()
    return analyzer.analyze_transcript(file_path, max_lines, sample_size)


def analyze_latest_transcripts(sample_size: int = 10) -> Dict[str, TranscriptStats]:
    """
    Convenience function to analyze the most recently created transcript directory.
    
    Args:
        sample_size: Number of sample lines to display per file
        
    Returns:
        Dictionary mapping filenames to analysis results
    """
    analyzer = SomaliTranscriptAnalyzer()
    return analyzer.analyze_directory()

#### 📋 INSTRUCTION
Run the cell below to visualize the transcript generated by Whisper.

In [ ]:
# Example usage
if __name__ == "__main__":
    # Example 1: Analyze specific transcript files
    transcript_files = [
        "transcripts/openai_whisper/IDAACADDA 01-JAN-2022.txt"
    ]
    
    for file_path in transcript_files:
        try:
            results = analyze_somali_transcript(file_path)
            print(f"\nAnalysis completed for: {file_path}")
            print(f"Quality score: {results} (if stats returned)")
        except Exception as e:
            print(f"Error analyzing {file_path}: {e}")
    
    # Example 2: Analyze all transcripts in the latest directory
    # all_results = analyze_latest_transcripts()
    # print(f"\nAnalyzed {len(all_results)} transcript files")
    
    # Example 3: Use the class directly for more control
    # analyzer = SomaliTranscriptAnalyzer()
    # results = analyzer.analyze_directory("transcripts_soundcloud_2025-03-15_to_2025-03-16")

### Whisper Small Somali (Specialized Model)

In [ ]:
"""
Somali-Specialized Whisper Audio Transcription System.

Uses HuggingFace Somali-trained Whisper model (steja/whisper-small-somali)
for improved Somali language recognition.
"""

import glob
import shutil
import subprocess
import logging
from datetime import datetime
from pathlib import Path
from typing import List, Optional, Union

import torch

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('somali_whisper_transcription.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


class SomaliWhisperTranscriber:
    """
    Specialized AudioTranscriber using HuggingFace Somali-trained Whisper model.
    
    Uses steja/whisper-small-somali for better Somali language recognition
    with GPU support and automatic fallback to multilingual models.
    """
    
    SUPPORTED_FORMATS = ["wav", "mp3", "m4a", "flac", "ogg", "webm", "mp4"]
    
    AVAILABLE_MODELS = {
        "somali": "steja/whisper-small-somali",
        "multilingual": "openai/whisper-small",
        "large": "openai/whisper-large-v3"
    }
    
    def __init__(
        self, 
        model_type: str = "somali", 
        output_base_dir: Optional[str] = None, 
        use_gpu: bool = True
    ) -> None:
        """
        Initialize the Somali-specialized Whisper transcriber.
        
        Args:
            model_type: Model variant ("somali", "multilingual", "large")
            output_base_dir: Base directory for transcripts (defaults to project structure)
            use_gpu: Whether to use GPU acceleration if available
            
        Raises:
            subprocess.CalledProcessError: If dependency installation fails
        """
        if model_type not in self.AVAILABLE_MODELS:
            logger.warning(f"Unknown model type '{model_type}'. Using 'somali' instead.")
            model_type = "somali"
            
        self.model_type = model_type
        self.model_id = self.AVAILABLE_MODELS[model_type]
        self.model = None
        self.processor = None
        self.transcriber = None
        self.use_gpu = use_gpu
        self.device = self._setup_device()
        
        if output_base_dir is None:
            # Separate directory for Somali-specialized transcripts
            self.output_base_dir = (
                self._find_project_root() / "data" / "02_intermediate" / 
                "transcripts" / "somali_whisper"
            )
        else:
            self.output_base_dir = Path(output_base_dir)
            
        self._ensure_dependencies()
    
    def _find_project_root(self) -> Path:
        """
        Find the project root directory by searching for project name.
        
        Returns:
            Path to project root directory
        """
        current = Path.cwd()
        project_name = "somali-radios-with-ai-for-food-security"
        
        while current != current.parent:
            if current.name == project_name:
                return current
            if (current / project_name).exists():
                return current / project_name
            current = current.parent
        
        project_root = Path.cwd() / project_name
        logger.info(f"Project root not found, using: {project_root}")
        return project_root
    
    def _ensure_dependencies(self) -> None:
        """Ensure all required dependencies are installed."""
        dependencies = ["transformers", "torch", "torchaudio", "soundfile"]
        for dep in dependencies:
            try:
                __import__(dep)
                logger.info(f"{dep} is already available")
            except ImportError:
                logger.info(f"Installing {dep}...")
                subprocess.run(["pip", "install", dep], check=True)
                logger.info(f"{dep} installed successfully")
    
    def _setup_device(self) -> str:
        """
        Setup computing device (GPU/CPU) for model.
        
        Returns:
            Device string ("cuda" or "cpu")
        """
        if self.use_gpu and torch.cuda.is_available():
            device = "cuda"
            gpu_name = torch.cuda.get_device_name(0)
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
            logger.info(f"GPU detected: {gpu_name} ({gpu_memory:.1f}GB)")
            logger.info(f"Using GPU acceleration")
        else:
            device = "cpu"
            if self.use_gpu:
                logger.warning("GPU requested but not available. Using CPU instead.")
            else:
                logger.info("Using CPU for transcription")
        
        return device
    
    def _load_model(self) -> None:
        """
        Load the HuggingFace Whisper model.
        
        Raises:
            Exception: If model loading fails and fallback also fails
        """
        if self.transcriber is None:
            logger.info(f"Loading HuggingFace model '{self.model_id}' on {self.device}")
            
            from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor
            
            try:
                self.model = AutoModelForSpeechSeq2Seq.from_pretrained(self.model_id)
                self.processor = AutoProcessor.from_pretrained(self.model_id)
                
                self.model.to(self.device)
                
                # Create pipeline with device-specific batch size
                self.transcriber = pipeline(
                    "automatic-speech-recognition",
                    model=self.model,
                    tokenizer=self.processor.tokenizer,
                    feature_extractor=self.processor.feature_extractor,
                    max_new_tokens=128,
                    chunk_length_s=30,
                    batch_size=16 if self.device == "cuda" else 8,
                    device=self.device
                )
                
                if self.device == "cuda":
                    gpu_memory_used = torch.cuda.memory_allocated() / 1024**3
                    logger.info(f"Model loaded on GPU. GPU memory used: {gpu_memory_used:.2f}GB")
                else:
                    logger.info("Model loaded on CPU")
                    
            except Exception as e:
                logger.error(f"Failed to load model {self.model_id}: {e}")
                # Fallback to multilingual model if Somali model fails
                if self.model_type == "somali":
                    logger.info("Falling back to multilingual model...")
                    self.model_id = self.AVAILABLE_MODELS["multilingual"]
                    self.model_type = "multilingual"
                    self._load_model()  # Recursive call for fallback
                else:
                    raise
    
    def _get_audio_files(
        self, 
        input_dir: Path, 
        audio_formats: Optional[List[str]] = None
    ) -> List[Path]:
        """
        Get all audio files in the specified directory.
        
        Args:
            input_dir: Directory containing audio files
            audio_formats: List of file extensions to search for
            
        Returns:
            Sorted list of audio file paths
        """
        if audio_formats is None:
            audio_formats = self.SUPPORTED_FORMATS
            
        audio_files = []
        for fmt in audio_formats:
            pattern = input_dir / f"*.{fmt}"
            audio_files.extend(glob.glob(str(pattern)))
            
        return [Path(f) for f in sorted(audio_files)]
    
    def _convert_audio_to_wav(self, audio_file: Path, temp_dir: Path) -> Optional[Path]:
        """
        Convert audio file to WAV format using torchaudio.
        
        Args:
            audio_file: Path to input audio file
            temp_dir: Directory for temporary converted file
            
        Returns:
            Path to converted WAV file, or None if conversion failed
        """
        temp_dir.mkdir(parents=True, exist_ok=True)
        temp_wav = temp_dir / f"{audio_file.stem}.wav"
        
        try:
            import torchaudio
            waveform, sample_rate = torchaudio.load(str(audio_file))
            torchaudio.save(str(temp_wav), waveform, sample_rate)
            logger.info(f"Successfully converted to WAV: {temp_wav}")
            return temp_wav
        except Exception as e:
            logger.error(f"Error converting {audio_file}: {e}")
            return None
    
    def transcribe_file(
        self, 
        audio_file: Path, 
        output_dir: Path, 
        language: str = "so"
    ) -> Optional[Path]:
        """
        Transcribe a single audio file using HuggingFace model.
        
        Args:
            audio_file: Path to audio file
            output_dir: Directory to save transcript
            language: Language code (default: "so" for Somali)
            
        Returns:
            Path to saved transcript file, or None if transcription failed
        """
        self._load_model()
        
        base_name = audio_file.stem
        transcript_file = output_dir / f"{base_name}.txt"
        file_format = audio_file.suffix[1:].lower()
        
        logger.info(f"Transcribing: {audio_file.name} (Format: {file_format}) on {self.device}")
        
        try:
            if self.device == "cuda":
                torch.cuda.empty_cache()  # Clear GPU memory before transcription
            
            result = self.transcriber(str(audio_file))
            
            # Extract text from result (handle different response formats)
            if isinstance(result, dict) and "text" in result:
                transcript_text = result["text"]
            else:
                transcript_text = str(result)
            
            with open(transcript_file, "w", encoding="utf-8") as f:
                f.write(transcript_text.strip())
            
            logger.info(f"Transcript saved: {transcript_file}")
            return transcript_file
            
        except Exception as e:
            logger.warning(f"Direct transcription failed for {audio_file.name}: {e}")
            
            # Try converting to WAV for non-WAV files
            if file_format != "wav":
                temp_dir = output_dir / "_temp_conversion"
                temp_wav = self._convert_audio_to_wav(audio_file, temp_dir)
                
                if temp_wav and temp_wav.exists():
                    try:
                        if self.device == "cuda":
                            torch.cuda.empty_cache()
                        
                        result = self.transcriber(str(temp_wav))
                        
                        if isinstance(result, dict) and "text" in result:
                            transcript_text = result["text"]
                        else:
                            transcript_text = str(result)
                        
                        with open(transcript_file, "w", encoding="utf-8") as f:
                            f.write(transcript_text.strip())
                        
                        logger.info(f"Transcript saved after conversion: {transcript_file}")
                        return transcript_file
                        
                    except Exception as retry_e:
                        logger.error(f"Transcription failed even after conversion: {retry_e}")
            
            return None
    
    def transcribe_single_file_path(
        self, 
        audio_file_path: Union[str, Path], 
        language: str = "so"
    ) -> Optional[Path]:
        """
        Transcribe a single audio file from a file path.
        
        Args:
            audio_file_path: Path to audio file (absolute or relative to project root)
            language: Language code for transcription (default: "so" for Somali)
            
        Returns:
            Path to the saved transcript file, or None if transcription failed
            
        Raises:
            FileNotFoundError: If audio file cannot be located
        """
        audio_path = Path(audio_file_path)
        project_root = self._find_project_root()
        project_name = "somali-radios-with-ai-for-food-security"
        
        # Handle absolute paths
        if audio_path.is_absolute():
            if not audio_path.exists():
                logger.error(f"Audio file not found: {audio_path}")
                return None
        else:
            # Handle relative paths - try multiple locations
            path_str = str(audio_path)
            resolved_path = None
            
            # Extract path after project name if present
            if project_name in path_str:
                parts = path_str.split(f"{project_name}/", 1)
                if len(parts) > 1:
                    relative_part = parts[1]
                    potential_path = project_root / relative_part
                    if potential_path.exists():
                        resolved_path = potential_path
            
            # Try as-is relative to current directory
            if not resolved_path and audio_path.exists():
                resolved_path = audio_path.resolve()
            
            # Try relative to project root / data / 01_raw
            if not resolved_path:
                filename = audio_path.name
                potential_path = project_root / "data" / "01_raw" / filename
                if potential_path.exists():
                    resolved_path = potential_path
            
            # Try from project root parent
            if not resolved_path:
                potential_path = project_root.parent / audio_path
                if potential_path.exists():
                    resolved_path = potential_path
            
            if resolved_path:
                audio_path = resolved_path
            else:
                logger.error(f"Audio file not found: {audio_file_path}")
                return None
        
        # Ensure output directory exists
        self.output_base_dir.mkdir(parents=True, exist_ok=True)
        
        # Transcribe the file
        transcript_file = self.transcribe_file(audio_path, self.output_base_dir, language)
        
        # Clean up temporary conversion directory if it exists
        temp_dir = self.output_base_dir / "_temp_conversion"
        if temp_dir.exists():
            try:
                shutil.rmtree(temp_dir)
                logger.info("Removed temporary conversion directory")
            except Exception as e:
                logger.warning(f"Failed to remove temporary directory: {e}")
        
        if transcript_file:
            logger.info(f"Transcript saved: {transcript_file}")
        else:
            logger.error(f"Failed to transcribe: {audio_path}")
        
        return transcript_file
    
    def transcribe_directory(
        self, 
        input_dir: Union[str, Path], 
        output_dir_name: Optional[str] = None,
        language: str = "so", 
        audio_formats: Optional[List[str]] = None
    ) -> List[Path]:
        """
        Transcribe all audio files in a directory.
        
        Args:
            input_dir: Directory containing audio files
            output_dir_name: Name for output subdirectory (auto-generated if None)
            language: Language code (default: "so" for Somali)
            audio_formats: List of audio formats to process
            
        Returns:
            List of paths to successfully created transcript files
        """
        input_path = Path(input_dir)
        
        if not input_path.exists():
            logger.error(f"Input directory does not exist: {input_path}")
            return []
        
        if output_dir_name is None:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            output_dir_name = f"batch_{self.model_type}_{input_path.name}_{timestamp}"
        
        output_dir = self.output_base_dir / output_dir_name
        output_dir.mkdir(parents=True, exist_ok=True)
        
        audio_files = self._get_audio_files(input_path, audio_formats)
        
        if not audio_files:
            logger.warning(f"No audio files found in {input_path}")
            return []
        
        logger.info(f"Found {len(audio_files)} audio files to transcribe")
        logger.info(f"Using Somali model: {self.model_id}")
        logger.info(f"Output directory: {output_dir}")
        
        transcript_files = []
        failed_files = []
        
        for i, audio_file in enumerate(audio_files, 1):
            logger.info(f"Processing {i}/{len(audio_files)}: {audio_file.name}")
            
            result = self.transcribe_file(audio_file, output_dir, language)
            if result:
                transcript_files.append(result)
            else:
                failed_files.append(audio_file)
        
        # Clean up temporary conversion directory
        temp_dir = output_dir / "_temp_conversion"
        if temp_dir.exists():
            try:
                shutil.rmtree(temp_dir)
                logger.info("Removed temporary conversion directory")
            except Exception as e:
                logger.warning(f"Failed to remove temporary directory: {e}")
        
        # Log summary
        logger.info(f"\n=== Somali Whisper Transcription Summary ===")
        logger.info(f"Model: {self.model_id}")
        logger.info(f"Total files processed: {len(audio_files)}")
        logger.info(f"Successfully transcribed: {len(transcript_files)}")
        logger.info(f"Failed transcriptions: {len(failed_files)}")
        
        if failed_files:
            logger.warning("Failed files:")
            for file_path in failed_files:
                logger.warning(f"  - {file_path}")
        
        return transcript_files
    
    def get_system_info(self) -> dict:
        """
        Get system information for GPU/CPU usage.
        
        Returns:
            Dictionary containing system and model information
        """
        info = {
            "transcriber_type": "HuggingFace Somali Whisper",
            "device": self.device,
            "model_type": self.model_type,
            "model_id": self.model_id,
            "torch_version": torch.__version__,
        }
        
        if torch.cuda.is_available():
            info.update({
                "cuda_available": True,
                "cuda_version": torch.version.cuda,
                "gpu_count": torch.cuda.device_count(),
                "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else None,
                "gpu_memory_total": f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB" if torch.cuda.device_count() > 0 else None
            })
        else:
            info["cuda_available"] = False
        
        return info


# Convenience functions
def transcribe_audio_file_somali(
    audio_file_path: Union[str, Path], 
    language: str = "so",
    model_type: str = "somali", 
    use_gpu: bool = True
) -> Optional[Path]:
    """
    Transcribe a single audio file using Somali specialized Whisper model.
    
    Args:
        audio_file_path: Path to audio file (absolute or relative)
        language: Language code for transcription (default: "so" for Somali)
        model_type: Model type to use ("somali", "multilingual", "large")
        use_gpu: Whether to use GPU acceleration if available
        
    Returns:
        Path to the saved transcript file, or None if transcription failed
    """
    transcriber = SomaliWhisperTranscriber(model_type=model_type, use_gpu=use_gpu)
    system_info = transcriber.get_system_info()
    logger.info(f"System Info: {system_info}")
    return transcriber.transcribe_single_file_path(audio_file_path, language)


def transcribe_audio_directory_somali(
    input_dir: Union[str, Path], 
    output_dir: Optional[str] = None, 
    language: str = "so", 
    audio_formats: Optional[List[str]] = None,
    model_type: str = "somali", 
    use_gpu: bool = True
) -> List[Path]:
    """
    Transcribe all audio files in a directory using Somali specialized Whisper.
    
    Args:
        input_dir: Directory containing audio files
        output_dir: Name for output subdirectory
        language: Language code (default: "so")
        audio_formats: List of audio formats to process
        model_type: Model type ("somali", "multilingual", "large")
        use_gpu: Whether to use GPU
        
    Returns:
        List of paths to created transcript files
    """
    transcriber = SomaliWhisperTranscriber(model_type=model_type, use_gpu=use_gpu)
    system_info = transcriber.get_system_info()
    logger.info(f"System Info: {system_info}")
    return transcriber.transcribe_directory(input_dir, output_dir, language, audio_formats)

In [ ]:
# Somali specialized approach - single file transcription
print("\n=== Using Somali Specialized Model ===")


audio_file_path = "somali-radios-with-ai-for-food-security/data/01_raw/IDAACADDA 01-JAN-2022.mp3"

transcript_path = transcribe_audio_file_somali(
    audio_file_path=audio_file_path,
    language="so",
    model_type="somali",
    use_gpu=True
)

if transcript_path:
    print(f"Somali Specialized Model completed! Transcript saved to: {transcript_path}")
else:
    print("Transcription failed!")

## Evaluation of Whisper Small Somali (Specialized Model)

In [ ]:
# Example usage
if __name__ == "__main__":
    # Example 1: Analyze specific transcript files
    transcript_files = [
        "transcripts/somali_whisper/IDAACADDA 01-JAN-2022.txt"
    ]
    
    for file_path in transcript_files:
        try:
            results = analyze_somali_transcript(file_path)
            print(f"\nAnalysis completed for: {file_path}")
            print(f"Quality score: {results} (if stats returned)")
        except Exception as e:
            print(f"Error analyzing {file_path}: {e}")
    
    # Example 2: Analyze all transcripts in the latest directory
    # all_results = analyze_latest_transcripts()
    # print(f"\nAnalyzed {len(all_results)} transcript files")
    
    # Example 3: Use the class directly for more control
    # analyzer = SomaliTranscriptAnalyzer()
    # results = analyzer.analyze_directory("transcripts_soundcloud_2025-03-15_to_2025-03-16")

## Gemini 2.5 Flash

In [ ]:
import os
import glob
import json
import time
import shutil
import subprocess
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from datetime import datetime
from google import genai
from google.genai import types


def load_api_key(env_file_path: str = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/.env") -> str:
    """
    Load API key from environment file.
    
    Args:
        env_file_path: Path to the .env file containing the API key
        
    Returns:
        str: The API key
        
    Raises:
        FileNotFoundError: If the .env file doesn't exist
        ValueError: If the API key is not found or is empty
    """
    if not os.path.exists(env_file_path):
        raise FileNotFoundError(f"Environment file not found: {env_file_path}")
    
    with open(env_file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith('GEMINI_API_KEY='):
                api_key = line.split('=', 1)[1].strip('"\'')
                if not api_key or api_key == "api_key":
                    raise ValueError("API key is empty or contains placeholder value. Please update your .env file.")
                return api_key
    
    raise ValueError("GEMINI_API_KEY not found in environment file")


def setup_ffmpeg() -> bool:
    """
    Check for ffmpeg installation and install if needed.
    
    Returns:
        bool: True if ffmpeg is available, False otherwise
    """
    try:
        subprocess.run(['ffmpeg', '-version'], 
                      stdout=subprocess.PIPE, 
                      stderr=subprocess.PIPE, 
                      check=True)
        print("✓ ffmpeg is available")
        return True
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("Installing ffmpeg...")
        try:
            subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True)
            subprocess.run(['sudo', 'apt-get', 'install', '-y', 'ffmpeg'], check=True)
            print("✓ ffmpeg installed successfully")
            return True
        except subprocess.CalledProcessError as e:
            print(f"✗ Failed to install ffmpeg: {e}")
            print("Please install ffmpeg manually: sudo apt-get install ffmpeg")
            return False


def convert_audio_to_mp3(input_path: str, output_path: str) -> bool:
    """
    Convert audio file to MP3 format optimized for speech recognition.
    
    Args:
        input_path: Path to input audio file
        output_path: Path for output MP3 file
        
    Returns:
        bool: True if conversion successful, False otherwise
    """
    try:
        cmd = [
            'ffmpeg', '-y', '-i', input_path,
            '-vn',  # no video
            '-acodec', 'libmp3lame',  # encode to mp3
            '-ar', '16000',  # 16 kHz sample rate (optimal for speech)
            '-ac', '1',  # mono audio
            '-b:a', '32k',  # bitrate for good quality/size balance
            output_path
        ]
        
        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True
        )
        return True
        
    except subprocess.CalledProcessError as e:
        print(f"✗ Conversion failed: {e.stderr.decode()}")
        return False


def setup_backup_directory(backup_path: str = "transcripts_backup") -> Optional[str]:
    """
    Create a backup directory for transcript files.
    
    Args:
        backup_path: Path for backup directory
        
    Returns:
        Optional[str]: Backup directory path if successful, None otherwise
    """
    try:
        backup_dir = Path(backup_path)
        backup_dir.mkdir(parents=True, exist_ok=True)
        print(f"📁 Backup directory ready: {backup_dir.absolute()}")
        return str(backup_dir.absolute())
    except Exception as e:
        print(f"✗ Failed to create backup directory: {e}")
        return None


def transcribe_single_audio(
    client: genai.Client,
    audio_path: str,
    model: str,
    language: str,
    retry_count: int,
    delay_between_failures: int
) -> Tuple[bool, str]:
    """
    Transcribe a single audio file with retry logic.
    
    Args:
        client: Initialized Gemini client
        audio_path: Path to audio file (should be MP3)
        model: Model name to use
        language: Language code for transcription
        retry_count: Number of retry attempts
        delay_between_failures: Seconds to wait between retries
        
    Returns:
        Tuple[bool, str]: (success, transcript_text_or_error_message)
    """
    for attempt in range(retry_count):
        try:
            with open(audio_path, "rb") as f:
                audio_bytes = f.read()

            audio_part = types.Part.from_bytes(
                data=audio_bytes,
                mime_type="audio/mp3"
            )

            # Enhanced prompt for better transcription quality
            prompt = (
                f"Please transcribe the following audio accurately in {language} language. "
                f"Maintain original speaker patterns and include natural pauses where appropriate."
            )

            response = client.models.generate_content(
                model=model,
                contents=[prompt, audio_part]
            )

            if hasattr(response, 'text') and response.text:
                return True, response.text
            else:
                return False, "No transcript text found in response"

        except Exception as e:
            error_msg = f"Attempt {attempt+1}/{retry_count} failed: {str(e)}"
            print(f"  {error_msg}")
            
            if attempt < retry_count - 1:
                print(f"  Waiting {delay_between_failures} seconds before retrying...")
                time.sleep(delay_between_failures)
            else:
                return False, f"All attempts failed. Last error: {str(e)}"
    
    return False, "Unknown error occurred"


def save_failure_log(output_dir: str, filename: str, file_path: str, error: str) -> None:
    """Save failure information to a log file."""
    log_file = os.path.join(output_dir, "failed_transcriptions.jsonl")
    failure_info = {
        "filename": filename,
        "path": file_path,
        "timestamp": datetime.now().isoformat(),
        "error": error
    }
    
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(failure_info) + "\n")


def copy_to_backup(transcript_files: List[str], output_dir: str, backup_path: str = "transcripts_backup") -> int:
    """
    Copy transcript files to backup directory.
    
    Args:
        transcript_files: List of transcript file paths
        output_dir: Original output directory
        backup_path: Backup directory path
    
    Returns:
        int: Number of files successfully copied
    """
    backup_dir_name = os.path.basename(output_dir.rstrip('/'))
    backup_full_path = Path(backup_path) / backup_dir_name
    
    print(f"📁 Saving transcripts to backup directory: {backup_full_path}")
    
    try:
        backup_full_path.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print(f"✗ Failed to create backup directory: {e}")
        return 0
    
    copied_count = 0
    for file_path in transcript_files:
        try:
            shutil.copy(file_path, backup_full_path / os.path.basename(file_path))
            copied_count += 1
        except Exception as e:
            print(f"✗ Failed to copy {file_path} to backup: {e}")
    
    print(f"✓ Successfully copied {copied_count} files to backup directory")
    return copied_count


def transcribe_audio_files(
    audio_file_path: str,
    output_dir: Optional[str] = None,
    language: str = "so",
    audio_formats: Optional[List[str]] = None,
    model: str = "gemini-2.5-flash-preview-0925",
    retry_count: int = 3,
    delay_between_failures: int = 10
) -> List[str]:
    """
    Transcribe audio using Google Gemini 2.5 Flash.
    Accepts either a single audio file path or a directory of audio files and
    always saves transcripts to the project's 02_intermediate/transcripts directory.

    Args:
        audio_file_path: Path to an audio file or directory containing audio files
        output_dir: Directory to save transcripts (auto-generated if None)
        language: Language code for transcription (e.g., "so" for Somali)
        audio_formats: List of audio formats to process when a directory is provided
        model: Gemini model to use (defaults to Gemini 2.5 Flash preview)
        retry_count: Number of retry attempts for failed transcriptions
        delay_between_failures: Seconds to wait between retry attempts

    Returns:
        List[str]: List of transcript file paths created

    Raises:
        FileNotFoundError: If the audio path doesn't exist
        EnvironmentError: If API key cannot be loaded
    """
    
    source_path = Path(audio_file_path)

    print("🎵 Starting audio transcription with Gemini 2.5 Flash")
    if source_path.is_dir():
        print(f"📂 Input directory: {source_path}")
    else:
        print(f"🎧 Input file: {source_path}")
    
    if not source_path.exists():
        raise FileNotFoundError(f"Audio path not found: {audio_file_path}")
    
    # Load API key
    try:
        api_key = load_api_key()
        print("🔑 API key loaded successfully")
    except Exception as e:
        raise EnvironmentError(f"Failed to load API key: {e}")
    
    # Setup dependencies
    if not setup_ffmpeg():
        print("⚠️  Warning: ffmpeg not available. Only MP3 files can be processed.")
    
    # Initialize Gemini client
    try:
        client = genai.Client(api_key=api_key)
        print(f"🤖 Using model: {model}")
    except Exception as e:
        raise EnvironmentError(f"Failed to initialize Gemini client: {e}")
    
    # Set defaults
    if audio_formats is None:
        audio_formats = ["wav", "mp3", "m4a", "flac", "ogg"]
    
    # Determine dataset name and output directory
    if not output_dir:
        if source_path.is_dir():
            dataset_name = source_path.name
        else:
            dataset_name = source_path.stem
        project_root = Path("/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security")
        output_dir = project_root / "data" / "02_intermediate" / "transcripts" / f"gemini_transcripts_{dataset_name}"
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    print(f"📁 Output directory: {output_dir}")
    
    # Assemble audio file list
    audio_files: List[str] = []
    if source_path.is_dir():
        for fmt in audio_formats:
            pattern = os.path.join(str(source_path), f"*.{fmt}")
            audio_files.extend(glob.glob(pattern))
        if not audio_files:
            print(f"❌ No audio files found with formats {audio_formats} in {source_path}")
            return []
    else:
        if source_path.suffix.replace(".", "").lower() not in [fmt.lower() for fmt in audio_formats]:
            print(f"⚠️  Provided file extension '{source_path.suffix}' not in allowed formats {audio_formats}. Continuing anyway.")
        audio_files = [str(source_path)]
    
    print(f"📊 Found {len(audio_files)} audio file{'s' if len(audio_files) != 1 else ''} to transcribe")
    print(f"🎯 Processing formats: {', '.join(audio_formats)}")
    print(f"🌍 Target language: {language}")
    
    # Temporary conversion directory sits next to provided path
    temp_dir_parent = source_path if source_path.is_dir() else source_path.parent
    temp_dir = temp_dir_parent / "_temp_conversion"
    temp_dir.mkdir(exist_ok=True)
    
    transcript_files = []
    processed_count = 0
    skipped_count = 0
    failed_count = 0
    
    try:
        for audio_file in audio_files:
            filename = os.path.basename(audio_file)
            base_name = Path(filename).stem
            file_extension = Path(filename).suffix[1:].lower()
            transcript_file = os.path.join(output_dir, f"{base_name}.txt")
            
            # Skip if already transcribed
            if os.path.exists(transcript_file):
                print(f"⏭️  Skipping {filename} - already transcribed")
                transcript_files.append(transcript_file)
                skipped_count += 1
                continue
            
            print(f"\n🎙️  Processing: {filename} ({file_extension.upper()})")
            
            # Handle format conversion
            current_audio_path = audio_file
            temp_file_created = False
            
            if file_extension != "mp3":
                temp_mp3_path = temp_dir / f"{base_name}.mp3"
                print(f"🔄 Converting to MP3...")
                
                if convert_audio_to_mp3(audio_file, str(temp_mp3_path)):
                    current_audio_path = str(temp_mp3_path)
                    temp_file_created = True
                    print("✓ Conversion successful")
                else:
                    print(f"✗ Conversion failed for {filename}")
                    save_failure_log(output_dir, filename, audio_file, "Audio format conversion failed")
                    failed_count += 1
                    continue
            
            # Transcribe audio
            print("🎯 Transcribing...")
            success, result = transcribe_single_audio(
                client, current_audio_path, model, language, 
                retry_count, delay_between_failures
            )
            
            if success:
                # Save transcript
                with open(transcript_file, "w", encoding="utf-8") as f:
                    f.write(result)
                
                transcript_files.append(transcript_file)
                processed_count += 1
                print(f"✅ Successfully transcribed {filename}")
            else:
                print(f"❌ Failed to transcribe {filename}")
                save_failure_log(output_dir, filename, audio_file, result)
                failed_count += 1
            
            # Clean up temporary file
            if temp_file_created and os.path.exists(current_audio_path):
                os.remove(current_audio_path)
    
    finally:
        # Clean up temporary directory
        if temp_dir.exists():
            try:
                shutil.rmtree(temp_dir)
                print(f"🧹 Cleaned up temporary files")
            except Exception as e:
                print(f"⚠️  Warning: Could not remove temp directory: {e}")
    
    # Print summary
    print(f"\n📈 Transcription Summary:")
    print(f"   Total files found: {len(audio_files)}")
    print(f"   Successfully transcribed: {processed_count}")
    print(f"   Already existed (skipped): {skipped_count}")
    print(f"   Failed: {failed_count}")
    print(f"   Output directory: {output_dir}")
    
    if failed_count > 0:
        print(f"   Check failed_transcriptions.jsonl for error details")
    
    return transcript_files

In [ ]:
# Usage example
if __name__ == "__main__":
    audio_file_path = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/IDAACADDA 01-JAN-2022.mp3"
    
    # Transcribe a single audio file
    # Output will automatically go to the project's 02_intermediate/transcripts directory
    transcripts = transcribe_audio_files(
        audio_file_path=audio_file_path,
        language="so",  # Somali language code
        model="gemini-2.5-flash"  # Latest Gemini 2.5 Flash model
    )
    
    print(f"\n🎉 Transcription complete! Created {len(transcripts)} transcript files.")
    print(f"📂 Files saved to the project's 02_intermediate/transcripts directory")

## Evaluation of Gemini 2.5 Flash

In [ ]:
# Example usage
if __name__ == "__main__":
    # Example 1: Analyze specific transcript files
    transcript_files = [
        "transcripts/gemini/IDAACADDA 01-JAN-2022.txt"
    ]
    
    for file_path in transcript_files:
        try:
            results = analyze_somali_transcript(file_path)
            print(f"\nAnalysis completed for: {file_path}")
            print(f"Quality score: {results} (if stats returned)")
        except Exception as e:
            print(f"Error analyzing {file_path}: {e}")
    
    # Example 2: Analyze all transcripts in the latest directory
    # all_results = analyze_latest_transcripts()
    # print(f"\nAnalyzed {len(all_results)} transcript files")
    
    # Example 3: Use the class directly for more control
    # analyzer = SomaliTranscriptAnalyzer()
    # results = analyzer.analyze_directory("transcripts_soundcloud_2025-03-15_to_2025-03-16")

In [ ]:
# # Example usage
# if __name__ == "__main__":
#     # Example 1: Analyze specific transcript files
#     transcript_files = [
#         "transcripts/gemini_transcripts_soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 15-MAR-2025.txt",
#         "transcripts/gemini_transcripts_soundcloud_2025-03-15_to_2025-03-16/IDAACADDA 16-MAR-2025.txt"
#     ]
    
#     for file_path in transcript_files:
#         try:
#             results = analyze_somali_transcript(file_path)
#             print(f"\nAnalysis completed for: {file_path}")
#             print(f"Quality score: {results} (if stats returned)")
#         except Exception as e:
#             print(f"Error analyzing {file_path}: {e}")
    
#     # Example 2: Analyze all transcripts in the latest directory
#     # all_results = analyze_latest_transcripts()
#     # print(f"\nAnalyzed {len(all_results)} transcript files")
    
#     # Example 3: Use the class directly for more control
#     # analyzer = SomaliTranscriptAnalyzer()
#     # results = analyzer.analyze_directory("transcripts_soundcloud_2025-03-15_to_2025-03-16")

## [Mustafaa4a/ASR-Somali](https://huggingface.co/Mustafaa4a/ASR-Somali)

In [ ]:
import os
from pathlib import Path
from typing import Dict, List
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import librosa
import soundfile as sf
from datetime import datetime

def setup_model(model_name: str = "Mustafaa4a/ASR-Somali") -> tuple:
    """
    Load the Wav2Vec2 model and processor for Somali ASR.
    
    Args:
        model_name: HuggingFace model identifier
        
    Returns:
        Tuple of (processor, model) ready for inference
        
    Raises:
        RuntimeError: If model fails to load
    """
    print(f"Loading model: {model_name}")
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    model = Wav2Vec2ForCTC.from_pretrained(model_name)
    
    # Move to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"Model loaded on: {device}")
    
    return processor, model, device

def load_and_preprocess_audio(audio_path: str, target_sr: int = 16000) -> torch.Tensor:
    """
    Load audio file and resample to target sampling rate.
    
    Args:
        audio_path: Path to MP3 audio file
        target_sr: Target sampling rate (Wav2Vec2 expects 16kHz)
        
    Returns:
        Audio tensor resampled to target_sr
        
    Raises:
        FileNotFoundError: If audio file doesn't exist
        RuntimeError: If audio loading fails
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    
    print(f"Loading audio: {audio_path}")
    # Load audio and resample to 16kHz (required by Wav2Vec2)
    audio, sr = librosa.load(audio_path, sr=target_sr)
    print(f"Audio duration: {len(audio) / sr:.2f} seconds")
    
    return audio

def transcribe_audio(
    audio: torch.Tensor,
    processor: Wav2Vec2Processor,
    model: Wav2Vec2ForCTC,
    device: str,
    chunk_length_s: int = 60
) -> str:
    """
    Transcribe audio using Wav2Vec2 model with chunking for long files.
    
    Args:
        audio: Audio tensor (16kHz sampling rate)
        processor: Wav2Vec2 processor for feature extraction
        model: Wav2Vec2 model for ASR
        device: Device to run inference on ('cuda' or 'cpu')
        chunk_length_s: Length of audio chunks in seconds (60s for GPU, 30s for CPU)
        
    Returns:
        Transcribed text
        
    Raises:
        RuntimeError: If transcription fails
    """
    # Process audio in chunks to handle long files
    sampling_rate = 16000
    chunk_length = chunk_length_s * sampling_rate
    transcriptions = []
    
    # Split audio into chunks
    for i in range(0, len(audio), chunk_length):
        chunk = audio[i:i + chunk_length]
        
        # Prepare input features
        input_values = processor(
            chunk,
            sampling_rate=sampling_rate,
            return_tensors="pt",
            padding=True
        ).input_values.to(device)
        
        # Perform inference
        with torch.no_grad():
            logits = model(input_values).logits
        
        # Decode predictions
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = processor.batch_decode(predicted_ids)[0]
        transcriptions.append(transcription)
        
        print(f"Processed chunk {i // chunk_length + 1}/{(len(audio) - 1) // chunk_length + 1}")
    
    # Combine all chunks
    full_transcription = " ".join(transcriptions)
    return full_transcription

def save_transcription(
    transcription: str,
    audio_filename: str,
    output_dir: str
) -> str:
    """
    Save transcription to text file with metadata.
    
    Args:
        transcription: Transcribed text
        audio_filename: Original audio filename (for naming output)
        output_dir: Directory to save transcription
        
    Returns:
        Path to saved transcription file
        
    Raises:
        IOError: If file writing fails
    """
    # Create output directory if it doesn't exist
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Generate output filename
    base_name = Path(audio_filename).stem
    output_path = os.path.join(output_dir, f"{base_name}_transcript.txt")
    
    # Save with metadata
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(f"# Transcription for: {audio_filename}\n")
        f.write(f"# Model: Mustafaa4a/ASR-Somali\n")
        f.write(f"# Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"# {'=' * 60}\n\n")
        f.write(transcription)
    
    print(f"Transcription saved to: {output_path}")
    return output_path

def transcribe_audio_files(
    audio_paths: List[str],
    output_dir: str,
    model_name: str = "Mustafaa4a/ASR-Somali"
) -> Dict[str, str]:
    """
    Transcribe multiple audio files and save results.
    
    Args:
        audio_paths: List of paths to audio files
        output_dir: Directory to save transcriptions
        model_name: HuggingFace model identifier
        
    Returns:
        Dictionary mapping audio paths to transcription file paths
        
    Raises:
        ValueError: If audio_paths is empty
    """
    if not audio_paths:
        raise ValueError("No audio paths provided")
    
    # Setup model once for all files
    processor, model, device = setup_model(model_name)
    
    results = {}
    
    for audio_path in audio_paths:
        try:
            print(f"\n{'=' * 80}")
            print(f"Processing: {audio_path}")
            print(f"{'=' * 80}")
            
            # Load and preprocess audio
            audio = load_and_preprocess_audio(audio_path)
            
            # Transcribe
            transcription = transcribe_audio(audio, processor, model, device)
            
            # Save transcription
            output_path = save_transcription(
                transcription,
                os.path.basename(audio_path),
                output_dir
            )
            
            results[audio_path] = output_path
            print(f"✓ Successfully transcribed: {os.path.basename(audio_path)}")
            
        except Exception as e:
            print(f"✗ Error processing {audio_path}: {str(e)}")
            results[audio_path] = None
    
    return results

In [ ]:
# Define audio files and output directory
audio_files = [
    "somali-radios-with-ai-for-food-security/data/01_raw/IDAACADDA 01-JAN-2022.mp3",
]

output_directory = "/somali-radios-with-ai-for-food-security/data/02_intermediate/transcripts/mustafaa4a_ASR-Somali"

# Run transcription
print("Starting Somali ASR Transcription Pipeline")
print(f"Number of files to process: {len(audio_files)}")

results = transcribe_audio_files(audio_files, output_directory)

# Summary
print(f"\n{'=' * 80}")
print("TRANSCRIPTION SUMMARY")
print(f"{'=' * 80}")
for audio_path, transcript_path in results.items():
    status = "✓ Success" if transcript_path else "✗ Failed"
    print(f"{status}: {os.path.basename(audio_path)}")
    if transcript_path:
        print(f"  → {transcript_path}")
print(f"{'=' * 80}")

### Evaluation of Mustafaa4a/ASR-Somali

In [ ]:
# Example usage
if __name__ == "__main__":
    # Example 1: Analyze specific transcript files
    transcript_files = [
        "transcripts/mustafaa4a_ASR-Somali_test/IDAACADDA 15-MAR-2025_transcript.txt",
        "transcripts/mustafaa4a_ASR-Somali_test/IDAACADDA 16-MAR-2025_transcript.txt"
    ]
    
    for file_path in transcript_files:
        try:
            results = analyze_somali_transcript(file_path)
            print(f"\nAnalysis completed for: {file_path}")
            print(f"Quality score: {results} (if stats returned)")
        except Exception as e:
            print(f"Error analyzing {file_path}: {e}")
    
    # Example 2: Analyze all transcripts in the latest directory
    # all_results = analyze_latest_transcripts()
    # print(f"\nAnalyzed {len(all_results)} transcript files")
    
    # Example 3: Use the class directly for more control
    # analyzer = SomaliTranscriptAnalyzer()
    # results = analyzer.analyze_directory("transcripts_soundcloud_2025-03-15_to_2025-03-16")

## [Scribe v1 from ElevenLabs](https://elevenlabs.io/speech-to-text/somali)

In [ ]:
import os
from pathlib import Path
from datetime import datetime
from elevenlabs import ElevenLabs

# Load ELEVEN_LABS_API_KEY from the project's .env file (without requiring python-dotenv)
ENV_PATH = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/.env"
API_KEY_VAR = "ELEVEN_LABS_API_KEY"

def load_elevenlabs_api_key(env_file_path: str = ENV_PATH) -> str:
    if not os.path.exists(env_file_path):
        raise FileNotFoundError(f"Environment file not found: {env_file_path}")
    with open(env_file_path, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith(f"{API_KEY_VAR}="):
                value = line.split("=", 1)[1].strip("\"' ")
                if not value or value == "your_eleven_labs_api_key":
                    raise ValueError(
                        f"{API_KEY_VAR} is empty or placeholder. Update your .env file."
                    )
                return value
    raise ValueError(f"{API_KEY_VAR} not found in environment file")

# Initialize ElevenLabs client
api_key = load_elevenlabs_api_key()
client = ElevenLabs(api_key=api_key)

# Input audio (absolute path provided by user)
audio_path = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/IDAACADDA 01-JAN-2022.mp3"
if not os.path.exists(audio_path):
    raise FileNotFoundError(f"Audio file not found: {audio_path}")

# Output path in the project transcripts directory
project_root = Path("/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security")
output_dir = project_root / "data" / "02_intermediate" / "transcripts" / "elevenlabs_scribe_v1"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / f"{Path(audio_path).stem}.txt"

# Perform transcription with Scribe v1
with open(audio_path, "rb") as f:
    result = client.speech_to_text.convert(
        file=f,
        model_id="scribe_v1",
    )

# Handle result and save transcript
text = None
try:
    # SDK may return an object with .text
    text = getattr(result, "text", None)
except Exception:
    text = None

if text is None:
    # Fallbacks if result is dict-like or plain string
    if isinstance(result, dict) and "text" in result:
        text = result["text"]
    elif isinstance(result, str):
        text = result

if not text:
    raise RuntimeError("No transcript text found in ElevenLabs response")

with open(output_file, "w", encoding="utf-8") as out:
    out.write(text.strip())

print(f"✅ ElevenLabs Scribe v1 transcription saved: {output_file}")


### Evaluation Scribe v1

In [ ]:
# Example usage
if __name__ == "__main__":
    # Example 1: Analyze specific transcript files
    transcript_files = [
        "transcripts/elevenlabs_scribe_v1/IDAACADDA 01-JAN-2022.txt"
    ]
    
    for file_path in transcript_files:
        try:
            results = analyze_somali_transcript(file_path)
            print(f"\nAnalysis completed for: {file_path}")
            print(f"Quality score: {results} (if stats returned)")
        except Exception as e:
            print(f"Error analyzing {file_path}: {e}")

# Conclusions 